In [1]:
# ============================================================
# CELL 1 — Imports
# All dependencies loaded once here; no re-imports in later cells
# ============================================================
import os
import json
import shutil
import random
import numpy as np
from collections import Counter

from ase.io import read                          # read QE .pwi input files
from pymatgen.core.structure import Structure    # pymatgen Structure object
from monty.serialization import loadfn, dumpfn  # convenient JSON I/O


In [2]:
# ============================================================
# CELL 2 — Isolated-atom reference energies → eref_90 dict
#
# Goal: build eref_90[label] = sum of isolated-atom energies (eV)
#       for each 90-atom (3×3×2) supercell composition.
#
# Why subtract isolated-atom energies?
#   NEP training uses cohesive/binding energies, not raw DFT total
#   energies.  Subtracting the sum of free-atom energies centres the
#   energy scale around zero and makes learning easier.
#
# Source of atom energies:
#   Single-atom spin-polarised QE calculations, stored in Ry.
#   Converted to eV with 1 Ry = 13.6057 eV (QE convention).
# ============================================================

# Raw isolated-atom total energies from QE (Ry)
energies_ry = {
    "Sb": -140.74358039,
    "Bi": -154.95167999,
    "Te": -174.52034190,
}

RY_TO_EV = 13.6057                                         # QE conversion factor
E_atom   = {k: v * RY_TO_EV for k, v in energies_ry.items()}  # eV

# Print converted values for reference
print("Isolated-atom energies:")
for el in energies_ry:
    print(f"  {el}: {energies_ry[el]:.8f} Ry  =  {E_atom[el]:.6f} eV")
print()

def calc_eref(n_Bi, n_Sb, n_Te):
    """Return sum of isolated-atom energies (eV) for a given composition."""
    return (
        n_Bi * E_atom["Bi"] +
        n_Sb * E_atom["Sb"] +
        n_Te * E_atom["Te"]
    )

# ------------------------------------------------------------------
# Build eref_90: read 90-atom QE input files to get composition,
# then compute the reference energy for each alloy endpoint.
#
# Note: Sb2Te3 has no 90-atom .pwi file, so it is added manually
#       (Bi:0, Sb:36, Te:54 → same 3×3×2 supercell logic).
# ------------------------------------------------------------------
folder  = "/home/ashwani/BiSbTe_AlloyTransport/BiSbTe_alloy_endpoints"
eref_90 = {}  # will be used in every subsequent cell

for fname in sorted(os.listdir(folder)):
    if not (fname.endswith(".pwi") and "primitive" in fname):
        continue
    atoms = read(os.path.join(folder, fname), format="espresso-in")
    if len(atoms) != 90:          # skip 5-atom primitive cells
        continue
    count = Counter(atoms.get_chemical_symbols())
    # derive clean label: e.g. "Bi2Te3", "BiSbTe20", ...
    label = (fname
             .replace("espresso_", "")
             .replace("_332_supercell_primitive.pwi", "")
             .replace("_primitive.pwi", ""))
    eref_90[label] = calc_eref(
        count.get("Bi", 0), count.get("Sb", 0), count.get("Te", 0)
    )

# Sb2Te3 endpoint added manually (no 90-atom .pwi exists)
eref_90["Sb2Te3"] = calc_eref(0, 36, 54)

print("eref_90 — 90-atom reference energies (eV):")
for label, eref in eref_90.items():
    print(f"  {label}: {eref:.4f} eV")


Isolated-atom energies:
  Sb: -140.74358039 Ry  =  -1914.914932 eV
  Bi: -154.95167999 Ry  =  -2108.226072 eV
  Te: -174.52034190 Ry  =  -2374.471416 eV

eref_90 — 90-atom reference energies (eV):
  Bi2Te3: -204117.5951 eV
  BiSbTe20: -202764.4171 eV
  BiSbTe40: -201411.2391 eV
  BiSbTe60: -199864.7500 eV
  BiSbTe80: -198511.5720 eV
  Sb2Te3: -197158.3940 eV


In [3]:
from ase.io import read
import numpy as np
import os

BASE_PATH = "/home/ashwani/BiSbTe_AlloyTransport/BiSbTe_alloy_endpoints"

def print_lattice_info(name, atoms):
    cell = atoms.get_cell()

    a = np.linalg.norm(cell[0])
    b = np.linalg.norm(cell[1])
    c = np.linalg.norm(cell[2])

    vol = atoms.get_volume()
    natoms = len(atoms)

    print(f"\n🔹 {name}")
    print(f"a = {a:.4f} Å | b = {b:.4f} Å | c = {c:.4f} Å")
    print(f"Volume = {vol:.4f} Å³")
    print(f"Atoms = {natoms}")
    print(f"Volume/atom = {vol/natoms:.4f} Å³")

# -------- FILES --------
files = {
    "Bi2Te3_primitive": "espresso_Bi2Te3_primitive.pwi",
    "Bi2Te3_conventional": "espresso_Bi2Te3_conventional.pwi",
    "Sb2Te3_primitive": "espresso_Sb2Te3_primitive.pwi",
    "Sb2Te3_conventional": "espresso_Sb2Te3_conventional.pwi",
    "Bi2Te3_332_supercell": "espresso_Bi2Te3_332_supercell_primitive.pwi",
    "BiSbTe20": "espresso_BiSbTe20_primitive.pwi",
    "BiSbTe40": "espresso_BiSbTe40_primitive.pwi",
    "BiSbTe60": "espresso_BiSbTe60_primitive.pwi",
    "BiSbTe80": "espresso_BiSbTe80_primitive.pwi",
}

# -------- PROCESS --------
for name, file in files.items():
    filepath = os.path.join(BASE_PATH, file)

    if not os.path.exists(filepath):
        print(f"❌ File not found: {filepath}")
        continue

    try:
        atoms = read(filepath, format='espresso-in')

        # Convert primitive → 3×3×2
        if "primitive" in name and "332" not in name:
            atoms = atoms.repeat((3, 3, 2))

        print_lattice_info(name, atoms)

    except Exception as e:
        print(f"❌ Error reading {file}: {e}")


🔹 Bi2Te3_primitive
a = 31.1775 Å | b = 31.1775 Å | c = 20.7850 Å
Volume = 3028.4444 Å³
Atoms = 90
Volume/atom = 33.6494 Å³

🔹 Bi2Te3_conventional
a = 4.3901 Å | b = 4.3901 Å | c = 30.2381 Å
Volume = 504.7106 Å³
Atoms = 15
Volume/atom = 33.6474 Å³

🔹 Sb2Te3_primitive
a = 30.8878 Å | b = 30.8878 Å | c = 20.5919 Å
Volume = 2789.4017 Å³
Atoms = 90
Volume/atom = 30.9934 Å³

🔹 Sb2Te3_conventional
a = 4.2439 Å | b = 4.2439 Å | c = 29.8758 Å
Volume = 465.9875 Å³
Atoms = 15
Volume/atom = 31.0658 Å³

🔹 Bi2Te3_332_supercell
a = 31.1775 Å | b = 31.1775 Å | c = 20.7850 Å
Volume = 3028.4444 Å³
Atoms = 90
Volume/atom = 33.6494 Å³

🔹 BiSbTe20
a = 31.1350 Å | b = 31.1385 Å | c = 20.7577 Å
Volume = 2983.9845 Å³
Atoms = 90
Volume/atom = 33.1554 Å³

🔹 BiSbTe40
a = 31.0767 Å | b = 31.0756 Å | c = 20.7174 Å
Volume = 2939.7254 Å³
Atoms = 90
Volume/atom = 32.6636 Å³

🔹 BiSbTe60
a = 30.9953 Å | b = 30.9967 Å | c = 20.6637 Å
Volume = 2888.6149 Å³
Atoms = 90
Volume/atom = 32.0957 Å³

🔹 BiSbTe80
a = 30.9394 Å | 

In [4]:
# # ============================================================
# # CELL 3 — Load all JSON datasets and subtract reference energies
# #
# # Three dataset types are included:
# #   1. random_disp_0.3A  — random atomic displacements (200 structs each)
# #   2. shear_strain      — shear-strained supercells (50 structs each)
# #   3. uniaxial_strain   — uniaxial strain along X/Y/Z (11 structs each)
# #
# # For each file label we map to the correct eref_90 key so the right
# # composition reference is subtracted from every structure.
# #
# # Output arrays (used in Cell 4 for XYZ writing):
# #   total_structures  : List[Structure]   — pymatgen Structure objects
# #   total_energies    : np.ndarray (eV)   — cohesive energies (ref subtracted)
# #   total_forces      : List[array]       — forces in eV/Å
# #   total_stresses    : List[array]       — virial stress in GPa (Voigt-6)
# # ============================================================

# # ------------------------------------------------------------------
# # Map each files{} label → eref_90 key
# # The tag after '/' in the label is matched by prefix against this dict.
# # Two naming conventions exist in the files dict:
# #   - random_disp uses "BiTeSb_XX"   → maps to "BiSbTeXX"
# #   - shear/uniaxial use bare "XX"   → maps to "BiSbTeXX"
# # ------------------------------------------------------------------
# label_map = {
#     "Bi2Te3"   : "Bi2Te3",
#     "BiTeSb_20": "BiSbTe20",
#     "BiTeSb_40": "BiSbTe40",
#     "BiTeSb_60": "BiSbTe60",
#     "BiTeSb_80": "BiSbTe80",
#     "Sb2Te3"   : "Sb2Te3",
#     "20"       : "BiSbTe20",
#     "40"       : "BiSbTe40",
#     "60"       : "BiSbTe60",
#     "80"       : "BiSbTe80",
# }

# def get_eref_key(file_label):
#     """Extract the eref_90 key from a files{} label string.
#     e.g. 'shear/20_1pct_0.3ptb' → tag='20_1pct_0.3ptb' → 'BiSbTe20'
#     """
#     tag = file_label.split("/")[-1]   # strip prefix like 'shear/'
#     for k, v in label_map.items():
#         if tag.startswith(k):
#             return v
#     raise ValueError(f"Cannot map '{file_label}' to an eref_90 key")

# def get_strain_metadata(file_label):
#     """
#     Parse strain value, perturbation amplitude, and strain direction
#     from the file label string.

#     Examples:
#       'shear/20_3pct_0.3ptb'      → strain_pct=3.0,  perturbation=0.3, direction=None
#       'shear/Bi2Te3_1pct_0ptb'    → strain_pct=1.0,  perturbation=0.0, direction=None
#       'uniaxial/60_X'             → strain_pct=None, perturbation=0.0, direction=X
#       'random_disp/BiTeSb_40'     → strain_pct=None, perturbation=0.3, direction=None
#     """
#     tag       = file_label.split("/")[-1]   # e.g. '20_3pct_0.3ptb', 'Bi2Te3_X'
#     dtype     = file_label.split("/")[0]    # 'random_disp', 'shear', 'uniaxial'

#     strain_pct    = None
#     perturbation  = 0.0
#     direction     = None

#     if dtype == "random_disp":
#         # random displacement, no strain, fixed 0.3 Å perturbation
#         perturbation = 0.3

#     elif dtype == "shear":
#         # e.g. "20_3pct_0.3ptb" or "Bi2Te3_1pct_0ptb"
#         import re
#         m_strain = re.search(r'(\d+)pct', tag)
#         m_ptb    = re.search(r'perturb_(\d+\.?\d*)|_(\d+\.?\d*)ptb', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         # parse perturbation from e.g. "0.3ptb" or "0ptb"
#         m_ptb2 = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb2:
#             perturbation = float(m_ptb2.group(1))

#     elif dtype == "uniaxial":
#         # e.g. "60_X" or "Bi2Te3_Y"
#         import re
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0   # uniaxial has no atomic perturbation

#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # File registry
# # Keys  = short human-readable labels (also used for eref mapping)
# # Values = full paths to JSON files produced by pymatgen/atomate
# # ------------------------------------------------------------------
# base_path = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/"

# files = {
#     # -------- random atomic displacements (0.3 Å amplitude) --------
#     "random_disp/Bi2Te3"        : base_path + "random_disp_0.3A/Bi2Te3.json",
#     "random_disp/BiTeSb_20"     : base_path + "random_disp_0.3A/BiTeSb_20.json",
#     "random_disp/BiTeSb_40"     : base_path + "random_disp_0.3A/BiTeSb_40.json",
#     "random_disp/BiTeSb_60"     : base_path + "random_disp_0.3A/BiTeSb_60.json",
#     "random_disp/BiTeSb_80"     : base_path + "random_disp_0.3A/BiTeSb_80.json",
#     "random_disp/Sb2Te3"        : base_path + "random_disp_0.3A/Sb2Te3.json",
#     # -------- shear strain (1% and 3%, with/without atomic perturbation) --------
#     "shear/Bi2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/Bi2Te3_1pct_0ptb"    : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0_ptb_lattice.json",
#     "shear/Bi2Te3_3pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0.3_ptb_lattice.json",
#     "shear/Bi2Te3_3pct_0ptb"    : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0_ptb_lattice.json",
#     "shear/20_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/20_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0_ptb_lattice.json",
#     "shear/20_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0.3_ptb_lattice.json",
#     "shear/20_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0_ptb_lattice.json",
#     "shear/40_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/40_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0_ptb_lattice.json",
#     "shear/40_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0.3_ptb_lattice.json",
#     "shear/40_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0_ptb_lattice.json",
#     "shear/60_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/60_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0_ptb_lattice.json",
#     "shear/60_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0.3_ptb_lattice.json",
#     "shear/60_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0_ptb_lattice.json",
#     "shear/80_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/80_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0_ptb_lattice.json",
#     "shear/80_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0.3_ptb_lattice.json",
#     "shear/80_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0_ptb_lattice.json",
#     "shear/Sb2Te3_1pct_0ptb"    : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0_ptb_lattice.json",
#     "shear/Sb2Te3_3pct_0ptb"    : base_path + "shear_strain/Sb2Te3_3_pct_perturb_0_ptb_lattice.json",
#     "shear/Sb2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0.3_ptb_lattice.json",
#     "shear/Sb2Te3_3pct_0.3ptb"  : base_path + "shear_strain/Sb2Te3_3_pct_perturb_0.3_ptb_lattice.json",
#     # -------- uniaxial strain along X, Y, Z --------
#     "uniaxial/Bi2Te3_X"         : base_path + "uniaxial_strain/Bi2Te3_X.json",
#     "uniaxial/Bi2Te3_Y"         : base_path + "uniaxial_strain/Bi2Te3_Y.json",
#     "uniaxial/Bi2Te3_Z"         : base_path + "uniaxial_strain/Bi2Te3_Z.json",
#     "uniaxial/20_X"             : base_path + "uniaxial_strain/BiSbTe_20_pct_X.json",
#     "uniaxial/20_Y"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Y.json",
#     "uniaxial/20_Z"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Z.json",
#     "uniaxial/40_X"             : base_path + "uniaxial_strain/BiSbTe_40_pct_X.json",
#     "uniaxial/40_Y"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Y.json",
#     "uniaxial/40_Z"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Z.json",
#     "uniaxial/60_X"             : base_path + "uniaxial_strain/BiSbTe_60_pct_X.json",
#     "uniaxial/60_Y"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Y.json",
#     "uniaxial/60_Z"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Z.json",
#     "uniaxial/80_X"             : base_path + "uniaxial_strain/BiSbTe_80_pct_X.json",
#     "uniaxial/80_Y"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Y.json",
#     "uniaxial/80_Z"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Z.json",
#     "uniaxial/Sb2Te3_X"         : base_path + "uniaxial_strain/Sb2Te3_X.json",
#     "uniaxial/Sb2Te3_Y"         : base_path + "uniaxial_strain/Sb2Te3_Y.json",
#     "uniaxial/Sb2Te3_Z"         : base_path + "uniaxial_strain/Sb2Te3_Z.json",
# }

# # ------------------------------------------------------------------
# # Load all JSON files, tag each entry with:
# #   _eref_key   → which eref_90 entry to subtract  (e.g. 'BiSbTe40')
# #   _file_label → full label for data_type parsing  (e.g. 'shear/40_1pct_0.3ptb')
# # Both tags are consumed in Cell 4 when building the entries list.
# # ------------------------------------------------------------------
# total_data = []
# print(f"{'File':<40} {'Structures':>12}")
# print("-" * 54)
# for label, path in files.items():
#     data     = loadfn(path)
#     eref_key = get_eref_key(label)
#     strain_pct, perturbation, direction = get_strain_metadata(label)
#     for d in data:
#         d["_eref_key"]    = eref_key      # composition key e.g. 'BiSbTe40'
#         d["_file_label"]  = label         # full label e.g. 'shear/20_3pct_0.3ptb'
#         d["_strain_pct"]  = strain_pct    # e.g. 3.0 or None
#         d["_perturbation"]= perturbation  # e.g. 0.3 or 0.0
#         d["_direction"]   = direction     # e.g. 'X' or None
#     print(f"{label:<40} {len(data):>12}")
#     total_data += data
    
# print("-" * 54)
# print(f"{'TOTAL':<40} {len(total_data):>12}")

# # ------------------------------------------------------------------
# # Extract arrays and subtract the composition-matched reference energy
# # from each structure's DFT total energy.
# #
# # total_energies = E_DFT - sum(n_X * E_atom[X])   (eV)
# #
# # Forces and stresses are NOT modified — they are already physical.
# # ------------------------------------------------------------------
# total_structures   = [d["structure"]                for d in total_data]
# total_forces       = [d["outputs"]["forces"]        for d in total_data]
# total_stresses     = [d["outputs"]["virial_stress"] for d in total_data]  # GPa, Voigt-6

# total_energies_raw = np.array([d["outputs"]["energy"]  for d in total_data])  # raw DFT (eV)
# total_eref         = np.array([eref_90[d["_eref_key"]] for d in total_data])  # reference (eV)
# total_energies     = total_energies_raw - total_eref   # cohesive energies for NEP training

# # Sanity check — should be small negative values (~-3 to -4 eV/atom × 90 atoms)
# print(f"\nEnergy after reference subtraction:")
# print(f"  min  : {total_energies.min():.4f} eV")
# print(f"  max  : {total_energies.max():.4f} eV")
# print(f"  mean : {total_energies.mean():.4f} eV")
# print(f"  mean/atom: {total_energies.mean()/90:.4f} eV/atom")

In [5]:
# ============================================================
# CELL 3 — Load all JSON datasets and subtract reference energies
#
# Three dataset types are included:
#   1. random_disp_0.3A  — random atomic displacements (200 structs each)
#   2. shear_strain      — shear-strained supercells (50 structs each)
#   3. uniaxial_strain   — uniaxial strain along X/Y/Z (11 structs each)
#   4. near_harmonic     — near-equilibrium harmonic sampling
#                          ├─ 90-atom primitive cells (Bi2Te3 × 120, alloys × 50 each)
#                          └─ 120-atom conventional cells (Bi2Te3 × 15, Sb2Te3 × 15)
#
# IMPORTANT — Two reference energies are needed for the near_harmonic group:
#
#   90-atom files  (Bi2Te3 + all alloys):
#     eref = eref_90[comp]                  ← same dict used by all other data types
#
#   120-atom files (Bi2Te3 & Sb2Te3 conventional cells):
#     eref = eref_90[comp] * (120 / 90)
#     Rationale: conventional and primitive cells share the same 2:3 stoichiometry
#     (48 Bi + 72 Te vs 36 Bi + 54 Te), so the total reference scales linearly.
#
# Output arrays (used in Cell 4 for XYZ writing):
#   total_structures  : List[Structure]   — pymatgen Structure objects
#   total_energies    : np.ndarray (eV)   — cohesive energies (ref subtracted)
#   total_forces      : List[array]       — forces in eV/Å
#   total_stresses    : List[array]       — virial stress in GPa (Voigt-6)
# ============================================================

# ------------------------------------------------------------------
# Map each files{} label → eref_90 key
# The tag after '/' in the label is matched by prefix against this dict.
# Two naming conventions exist in the files dict:
#   - random_disp uses "BiTeSb_XX"   → maps to "BiSbTeXX"
#   - shear/uniaxial use bare "XX"   → maps to "BiSbTeXX"
#   - near_harmonic: 90-atom uses comp name directly,
#                    120-atom uses comp name + scale flag
# ------------------------------------------------------------------
label_map = {
    "Bi2Te3"   : "Bi2Te3",
    "BiTeSb_20": "BiSbTe20",
    "BiTeSb_40": "BiSbTe40",
    "BiTeSb_60": "BiSbTe60",
    "BiTeSb_80": "BiSbTe80",
    "Sb2Te3"   : "Sb2Te3",
    "20"       : "BiSbTe20",
    "40"       : "BiSbTe40",
    "60"       : "BiSbTe60",
    "80"       : "BiSbTe80",
}

def get_eref_key(file_label):
    """Extract the eref_90 key from a files{} label string.
    e.g. 'shear/20_1pct_0.3ptb' → tag='20_1pct_0.3ptb' → 'BiSbTe20'
    """
    tag = file_label.split("/")[-1]   # strip prefix like 'shear/'
    for k, v in label_map.items():
        if tag.startswith(k):
            return v
    raise ValueError(f"Cannot map '{file_label}' to an eref_90 key")

def get_strain_metadata(file_label):
    """
    Parse strain value, perturbation amplitude, and strain direction
    from the file label string.

    Examples:
      'shear/20_3pct_0.3ptb'      → strain_pct=3.0,  perturbation=0.3, direction=None
      'shear/Bi2Te3_1pct_0ptb'    → strain_pct=1.0,  perturbation=0.0, direction=None
      'uniaxial/60_X'             → strain_pct=None, perturbation=0.0, direction=X
      'random_disp/BiTeSb_40'     → strain_pct=None, perturbation=0.3, direction=None
      'near_harmonic/Bi2Te3_90'   → strain_pct=None, perturbation=0.0, direction=None
      'near_harmonic/Bi2Te3_120'  → strain_pct=None, perturbation=0.0, direction=None
    """
    tag       = file_label.split("/")[-1]   # e.g. '20_3pct_0.3ptb', 'Bi2Te3_X'
    dtype     = file_label.split("/")[0]    # 'random_disp', 'shear', 'uniaxial', 'near_harmonic'

    strain_pct    = None
    perturbation  = 0.0
    direction     = None

    if dtype == "random_disp":
        # random displacement, no strain, fixed 0.3 Å perturbation
        perturbation = 0.3

    elif dtype == "shear":
        # e.g. "20_3pct_0.3ptb" or "Bi2Te3_1pct_0ptb"
        import re
        m_strain = re.search(r'(\d+)pct', tag)
        m_ptb    = re.search(r'perturb_(\d+\.?\d*)|_(\d+\.?\d*)ptb', tag)
        if m_strain:
            strain_pct = float(m_strain.group(1))
        # parse perturbation from e.g. "0.3ptb" or "0ptb"
        m_ptb2 = re.search(r'_(\d+\.?\d*)ptb', tag)
        if m_ptb2:
            perturbation = float(m_ptb2.group(1))

    elif dtype == "uniaxial":
        # e.g. "60_X" or "Bi2Te3_Y"
        import re
        m_dir = re.search(r'_(X|Y|Z)$', tag)
        if m_dir:
            direction = m_dir.group(1)
        perturbation = 0.0   # uniaxial has no atomic perturbation

    elif dtype == "near_harmonic":
        # no strain, no perturbation, no direction — all defaults
        perturbation = 0.0

    return strain_pct, perturbation, direction

# ------------------------------------------------------------------
# File registry
# Keys  = short human-readable labels (also used for eref mapping)
# Values = full paths to JSON files produced by pymatgen/atomate
# ------------------------------------------------------------------
base_path = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/"

files = {
    # -------- random atomic displacements (0.3 Å amplitude) --------
    "random_disp/Bi2Te3"        : base_path + "random_disp_0.3A/Bi2Te3.json",
    "random_disp/BiTeSb_20"     : base_path + "random_disp_0.3A/BiTeSb_20.json",
    "random_disp/BiTeSb_40"     : base_path + "random_disp_0.3A/BiTeSb_40.json",
    "random_disp/BiTeSb_60"     : base_path + "random_disp_0.3A/BiTeSb_60.json",
    "random_disp/BiTeSb_80"     : base_path + "random_disp_0.3A/BiTeSb_80.json",
    "random_disp/Sb2Te3"        : base_path + "random_disp_0.3A/Sb2Te3.json",
    # -------- shear strain (1% and 3%, with/without atomic perturbation) --------
    "shear/Bi2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/Bi2Te3_1pct_0ptb"    : base_path + "shear_strain/Bi2Te3_1_pct_perturb_0_ptb_lattice.json",
    "shear/Bi2Te3_3pct_0.3ptb"  : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/Bi2Te3_3pct_0ptb"    : base_path + "shear_strain/Bi2Te3_3_pct_perturb_0_ptb_lattice.json",
    "shear/20_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/20_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_1_pct_perturb_0_ptb_lattice.json",
    "shear/20_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/20_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_20_3_pct_perturb_0_ptb_lattice.json",
    "shear/40_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/40_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_1_pct_perturb_0_ptb_lattice.json",
    "shear/40_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/40_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_40_3_pct_perturb_0_ptb_lattice.json",
    "shear/60_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/60_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_1_pct_perturb_0_ptb_lattice.json",
    "shear/60_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/60_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_60_3_pct_perturb_0_ptb_lattice.json",
    "shear/80_1pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/80_1pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_1_pct_perturb_0_ptb_lattice.json",
    "shear/80_3pct_0.3ptb"      : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0.3_ptb_lattice.json",
    "shear/80_3pct_0ptb"        : base_path + "shear_strain/BiSBTe_80_3_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_1pct_0ptb"    : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_3pct_0ptb"    : base_path + "shear_strain/Sb2Te3_3_pct_perturb_0_ptb_lattice.json",
    "shear/Sb2Te3_1pct_0.3ptb"  : base_path + "shear_strain/Sb2Te3_1_pct_perturb_0.3_ptb_lattice.json",
    "shear/Sb2Te3_3pct_0.3ptb"  : base_path + "shear_strain/Sb2Te3_3_pct_perturb_0.3_ptb_lattice.json",
    # -------- uniaxial strain along X, Y, Z --------
    "uniaxial/Bi2Te3_X"         : base_path + "uniaxial_strain/Bi2Te3_X.json",
    "uniaxial/Bi2Te3_Y"         : base_path + "uniaxial_strain/Bi2Te3_Y.json",
    "uniaxial/Bi2Te3_Z"         : base_path + "uniaxial_strain/Bi2Te3_Z.json",
    "uniaxial/20_X"             : base_path + "uniaxial_strain/BiSbTe_20_pct_X.json",
    "uniaxial/20_Y"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Y.json",
    "uniaxial/20_Z"             : base_path + "uniaxial_strain/BiSbTe_20_pct_Z.json",
    "uniaxial/40_X"             : base_path + "uniaxial_strain/BiSbTe_40_pct_X.json",
    "uniaxial/40_Y"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Y.json",
    "uniaxial/40_Z"             : base_path + "uniaxial_strain/BiSbTe_40_pct_Z.json",
    "uniaxial/60_X"             : base_path + "uniaxial_strain/BiSbTe_60_pct_X.json",
    "uniaxial/60_Y"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Y.json",
    "uniaxial/60_Z"             : base_path + "uniaxial_strain/BiSbTe_60_pct_Z.json",
    "uniaxial/80_X"             : base_path + "uniaxial_strain/BiSbTe_80_pct_X.json",
    "uniaxial/80_Y"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Y.json",
    "uniaxial/80_Z"             : base_path + "uniaxial_strain/BiSbTe_80_pct_Z.json",
    "uniaxial/Sb2Te3_X"         : base_path + "uniaxial_strain/Sb2Te3_X.json",
    "uniaxial/Sb2Te3_Y"         : base_path + "uniaxial_strain/Sb2Te3_Y.json",
    "uniaxial/Sb2Te3_Z"         : base_path + "uniaxial_strain/Sb2Te3_Z.json",
    # -------- near-harmonic sampling --------
    # 90-atom primitive cells — same eref_90 as all other data
    "near_harmonic/Bi2Te3"      : base_path + "near_harmonic/Bi2Te3_near_equib.json",
    "near_harmonic/Sb2Te3"      : base_path + "near_harmonic/Sb2Te3_near_equib.json",
    "near_harmonic/BiTeSb_20"   : base_path + "near_harmonic/BiSBTe_20_near_equib.json",
    "near_harmonic/BiTeSb_40"   : base_path + "near_harmonic/BiSBTe_40_near_equib.json",
    "near_harmonic/BiTeSb_60"   : base_path + "near_harmonic/BiSBTe_60_near_equib.json",
    "near_harmonic/BiTeSb_80"   : base_path + "near_harmonic/BiSBTe_80_near_equib.json",
    # 120-atom conventional cells — eref scaled by 120/90 (see note at top)
    "near_harmonic/Bi2Te3_conv120" : base_path + "near_harmonic/Bi2Te3_conventional_cell.json",
    "near_harmonic/Sb2Te3_conv120" : base_path + "near_harmonic/Sb2Te3_conventional_cell.json",
}

# ------------------------------------------------------------------
# Reference energy for 120-atom conventional cells
# These labels are NOT in label_map (which covers 90-atom cells only).
# We detect them by the '_conv120' suffix and scale eref_90 accordingly.
# ------------------------------------------------------------------
CONV120_LABELS = {
    "near_harmonic/Bi2Te3_conv120": "Bi2Te3",
    "near_harmonic/Sb2Te3_conv120": "Sb2Te3",
}
CONV120_SCALE  = 120 / 90   # = 4/3  (both comps have identical 2:3 stoichiometry)

def get_eref_for_label(file_label, eref_90_dict):
    """
    Return the correct reference energy for a given file_label.

    For 90-atom files → eref_90[comp]                  (same as existing data)
    For 120-atom conventional cells → eref_90[comp] * (120/90)
    """
    if file_label in CONV120_LABELS:
        comp = CONV120_LABELS[file_label]
        return eref_90_dict[comp] * CONV120_SCALE
    else:
        return eref_90_dict[get_eref_key(file_label)]

# ------------------------------------------------------------------
# Load all JSON files, tag each entry with:
#   _eref_key   → which eref_90 entry to subtract  (e.g. 'BiSbTe40')
#   _file_label → full label for data_type parsing  (e.g. 'shear/40_1pct_0.3ptb')
# Both tags are consumed in Cell 4 when building the entries list.
# ------------------------------------------------------------------
total_data = []
print(f"{'File':<45} {'Structures':>12} {'n_atoms':>8}")
print("-" * 67)
for label, path in files.items():
    data     = loadfn(path)
    strain_pct, perturbation, direction = get_strain_metadata(label)
    n_atoms  = len(data[0]["structure"])  # pymatgen Structure len = number of sites

    # assign eref_key (or None for conv120 — resolved at energy subtraction time)
    if label in CONV120_LABELS:
        eref_key = CONV120_LABELS[label]   # store comp key; scaling applied below
    else:
        eref_key = get_eref_key(label)

    for d in data:
        d["_eref_key"]      = eref_key
        d["_file_label"]    = label
        d["_strain_pct"]    = strain_pct
        d["_perturbation"]  = perturbation
        d["_direction"]     = direction
        d["_conv120"]       = (label in CONV120_LABELS)   # flag for energy subtraction
    print(f"{label:<45} {len(data):>12} {n_atoms:>8}")
    total_data += data

print("-" * 67)
print(f"{'TOTAL':<45} {len(total_data):>12}")

# ------------------------------------------------------------------
# Extract arrays and subtract the composition-matched reference energy
# from each structure's DFT total energy.
#
# total_energies = E_DFT - eref   (eV)
#
# For 90-atom cells:    eref = eref_90[comp]
# For 120-atom cells:   eref = eref_90[comp] * (120/90)
#
# Forces and stresses are NOT modified — they are already physical.
# ------------------------------------------------------------------
total_structures   = [d["structure"]                for d in total_data]
total_forces       = [d["outputs"]["forces"]        for d in total_data]
total_stresses     = [d["outputs"]["virial_stress"] for d in total_data]  # GPa, Voigt-6

total_energies_raw = np.array([d["outputs"]["energy"] for d in total_data])  # raw DFT (eV)

total_eref = np.array([
    eref_90[d["_eref_key"]] * (CONV120_SCALE if d["_conv120"] else 1.0)
    for d in total_data
])

total_energies = total_energies_raw - total_eref   # cohesive energies for NEP training

# Sanity check — near_harmonic cohesive energies should also be small negative values
print(f"\nEnergy after reference subtraction (ALL data):")
print(f"  min  : {total_energies.min():.4f} eV")
print(f"  max  : {total_energies.max():.4f} eV")
print(f"  mean : {total_energies.mean():.4f} eV")

# Per-type check: isolate near_harmonic cohesive energies
nh_mask     = np.array([d["_file_label"].startswith("near_harmonic") for d in total_data])
nh90_mask   = nh_mask & ~np.array([d["_conv120"] for d in total_data])
nh120_mask  = nh_mask &  np.array([d["_conv120"] for d in total_data])

if nh90_mask.any():
    e90  = total_energies[nh90_mask]
    print(f"\nnear_harmonic 90-atom cohesive energies:")
    print(f"  min={e90.min():.4f}  max={e90.max():.4f}  mean/atom={e90.mean()/90:.4f} eV/atom")

if nh120_mask.any():
    e120 = total_energies[nh120_mask]
    print(f"\nnear_harmonic 120-atom cohesive energies:")
    print(f"  min={e120.min():.4f}  max={e120.max():.4f}  mean/atom={e120.mean()/120:.4f} eV/atom")

File                                            Structures  n_atoms
-------------------------------------------------------------------
random_disp/Bi2Te3                                     200       90
random_disp/BiTeSb_20                                  200       90
random_disp/BiTeSb_40                                  200       90
random_disp/BiTeSb_60                                  200       90
random_disp/BiTeSb_80                                  200       90
random_disp/Sb2Te3                                     200       90
shear/Bi2Te3_1pct_0.3ptb                                50       90
shear/Bi2Te3_1pct_0ptb                                  50       90
shear/Bi2Te3_3pct_0.3ptb                                50       90
shear/Bi2Te3_3pct_0ptb                                  50       90
shear/20_1pct_0.3ptb                                    50       90
shear/20_1pct_0ptb                                      50       90
shear/20_3pct_0.3ptb                            

In [6]:
# ============================================================
# CELL 4 — stress convert + stratified split +
#           equilibrium structures — NO weights
# ============================================================

import re
import random
import os
from collections import defaultdict
from monty.serialization import loadfn

# ------------------------------------------------------------------
# Constants
# ------------------------------------------------------------------
FLIP_STRESS_SIGN  = True
GPA_TO_EV_PER_A3  = 1.0 / 160.21766208
N_EQ_COPIES       = 2     # repeat each eq structure N times

# ------------------------------------------------------------------
# 1. Stress conversion — GPa Voigt-6 → eV/Å³ 9-component
# ------------------------------------------------------------------
def convert_voigt6_gpa_to_ev_per_a3(voigt6):
    factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
    σxx, σyy, σzz, σyz, σxz, σxy = voigt6
    return [
        σxx*factor, σxy*factor, σxz*factor,
        σxy*factor, σyy*factor, σyz*factor,
        σxz*factor, σyz*factor, σzz*factor,
    ]

# ------------------------------------------------------------------
# 2. Metadata parser
# ------------------------------------------------------------------
def get_strain_metadata(file_label):
    tag   = file_label.split("/")[-1]
    dtype = file_label.split("/")[0]
    strain_pct   = None
    perturbation = 0.0
    direction    = None
    if dtype == "random_disp":
        perturbation = 0.3
    elif dtype == "shear":
        m_strain = re.search(r'(\d+)pct', tag)
        if m_strain:
            strain_pct = float(m_strain.group(1))
        m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
        if m_ptb:
            perturbation = float(m_ptb.group(1))
    elif dtype == "uniaxial":
        m_dir = re.search(r'_(X|Y|Z)$', tag)
        if m_dir:
            direction = m_dir.group(1)
        perturbation = 0.0
    elif dtype == "near_harmonic":
        perturbation = 0.0   # no perturbation, no strain, no direction
    return strain_pct, perturbation, direction

# ------------------------------------------------------------------
# 3. Sanity check on Cell 3 outputs
# ------------------------------------------------------------------
N = len(total_structures)
assert len(total_energies) == N
assert len(total_forces)   == N
assert len(total_stresses) == N
print(f"Cell 3 data: {N} structures OK")

# ------------------------------------------------------------------
# 4. Pack entries from Cell 3 data
# ------------------------------------------------------------------
entries = []
for i, d in enumerate(total_data):
    config_type                         = d["_eref_key"]
    data_type                           = d["_file_label"].split("/")[0]
    strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
    entries.append({
        "structure"   : total_structures[i],
        "_file_label" : d["_file_label"],
        "_conv120"    : d["_conv120"],          # True for 120-atom conventional cells
        "config_type" : config_type,
        "data_type"   : data_type,
        "strain_pct"  : strain_pct,
        "perturbation": perturbation,
        "direction"   : direction,
        "outputs": {
            "energy": total_energies[i],
            "forces": total_forces[i],
            "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])
        }
    })

# ------------------------------------------------------------------
# 5. Stratified 80/20 split — 20% from each individual file
# ------------------------------------------------------------------
random.seed(42)

strata = defaultdict(list)
for e in entries:
    strata[e["_file_label"]].append(e)

train_entries = []
test_entries  = []

print(f"\n{'File Label':<45} {'Total':>6} {'Train':>6} {'Test':>6}")
print("-" * 63)

for key in sorted(strata.keys()):
    group = strata[key]
    random.shuffle(group)
    n_test  = max(1, int(round(len(group) * 0.20)))
    n_train = len(group) - n_test
    test_entries  += group[:n_test]
    train_entries += group[n_test:]
    print(f"{key:<45} {len(group):>6} {n_train:>6} {n_test:>6}")

print("-" * 63)
print(f"{'TOTAL':<45} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
      f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# ------------------------------------------------------------------
# 6. Load equilibrium structures
#    Each comp gets its OWN eref_90 subtracted
# ------------------------------------------------------------------
eq_base  = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/equib_structure/"

eq_files = {
    "Bi2Te3"  : eq_base + "Bi2Te3_equilibrium.json",
    "BiSbTe20": eq_base + "BiSbTe20_equilibrium.json",
    "BiSbTe40": eq_base + "BiSbTe40_equilibrium.json",
    "BiSbTe60": eq_base + "BiSbTe60_equilibrium.json",
    "BiSbTe80": eq_base + "BiSbTe80_equilibrium.json",
    "Sb2Te3"  : eq_base + "Sb2Te3_equilibrium.json",
}

eq_entries = []
print(f"\n--- Equilibrium reference subtraction ---")
print(f"{'Comp':<12} {'E_raw(eV)':>14} {'eref_90(eV)':>14} {'E_coh(eV)':>12} {'eV/atom':>10}")
print("-" * 66)

for comp, path in eq_files.items():
    data   = loadfn(path)
    d      = data[0]
    struct = d["structure"]
    e_raw  = d["outputs"]["energy"]
    e_ref  = eref_90[comp]
    e_coh  = e_raw - e_ref
    forces = d["outputs"]["forces"]
    stress = convert_voigt6_gpa_to_ev_per_a3(d["outputs"]["virial_stress"])

    eq_entries.append({
        "structure"   : struct,
        "_file_label" : f"equilibrium/{comp}",
        "_conv120"    : False,
        "config_type" : comp,
        "data_type"   : "equilibrium",
        "strain_pct"  : None,
        "perturbation": 0.0,
        "direction"   : None,
        "outputs"     : {"energy": e_coh, "forces": forces, "stress": stress},
    })
    print(f"{comp:<12} {e_raw:>14.4f} {e_ref:>14.4f} {e_coh:>12.4f} {e_coh/len(struct):>10.4f}")

print("-" * 66)
eq_mean    = sum(e['outputs']['energy']/len(e['structure']) for e in eq_entries)/len(eq_entries)
# mean/atom for non-conv120 train entries (all 90-atom)
n90_train  = [e for e in train_entries if not e["_conv120"]]
train_mean = (sum(e['outputs']['energy'] for e in n90_train) / len(n90_train) / 90) if n90_train else float('nan')
print(f"Eq mean    : {eq_mean:.4f} eV/atom")
print(f"Train mean (90-atom) : {train_mean:.4f} eV/atom")

eq_train = eq_entries * N_EQ_COPIES
print(f"\n{len(eq_entries)} eq structs × {N_EQ_COPIES} copies = {len(eq_train)} for train")

# ------------------------------------------------------------------
# 7. XYZ writer — NO weight keyword in header
# ------------------------------------------------------------------
def write_entry_xyz(f, struct, out, config_type, data_type,
                    strain_pct, perturbation, direction):
    n = len(struct)
    f.write(f"{n}\n")
    lat_str    = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
    e_str      = f"{out['energy']:.12f}"
    s_str      = " ".join(f"{x:.12f}" for x in out['stress'])
    strain_str = f"strain_pct={strain_pct}"  if strain_pct is not None else "strain_pct=None"
    ptb_str    = f"perturbation={perturbation}A"
    dir_str    = f"direction={direction}"     if direction  is not None else "direction=None"
    header = [
        f'lattice="{lat_str}"',
        f'energy={e_str}',
        f'stress="{s_str}"',
        f'config_type={config_type}',
        f'data_type={data_type}',
        strain_str, ptb_str, dir_str,
        'energy_units=eV',
        'forces_units=eV/Ang',
        'stress_units=eV/Ang3',
        'properties=species:S:1:pos:R:3:forces:R:3'
    ]
    f.write(" ".join(header) + "\n")
    coords = struct.cart_coords
    for idx, (fx, fy, fz) in enumerate(out['forces']):
        sym      = struct.sites[idx].species.elements[0].symbol
        x, y, z  = coords[idx]
        f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")

# ------------------------------------------------------------------
# 8. Dataset writer
# ------------------------------------------------------------------
def write_dataset(fname, data, include_eq=False):
    with open(fname, 'w') as f:
        for e in data:
            write_entry_xyz(
                f, e['structure'], e['outputs'],
                e['config_type'], e['data_type'],
                e['strain_pct'],  e['perturbation'], e['direction'],
            )
        if include_eq:
            for e in eq_train:
                write_entry_xyz(
                    f, e['structure'], e['outputs'],
                    e['config_type'], e['data_type'],
                    e['strain_pct'],  e['perturbation'], e['direction'],
                )
    total = len(data) + (len(eq_train) if include_eq else 0)
    print(f"Wrote {total} entries → '{fname}'")

# ------------------------------------------------------------------
# 9. Write files
# ------------------------------------------------------------------
out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"
os.makedirs(out_dir, exist_ok=True)

write_dataset(f"{out_dir}/train.xyz", train_entries, include_eq=True)
write_dataset(f"{out_dir}/test.xyz",  test_entries,  include_eq=False)

# ------------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------------
n_random        = sum(1 for e in train_entries if e["data_type"]=="random_disp")
n_shear_perturb = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]!=0.0)
n_shear_clean   = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]==0.0)
n_uniaxial      = sum(1 for e in train_entries if e["data_type"]=="uniaxial")
n_nh_90         = sum(1 for e in train_entries if e["data_type"]=="near_harmonic" and not e["_conv120"])
n_nh_120        = sum(1 for e in train_entries if e["data_type"]=="near_harmonic" and     e["_conv120"])

print(f"\n=== DATASET COMPOSITION (no weights — all equal) ===")
print(f"{'Data type':<30} {'Structs':>8} {'Fraction':>10}")
print("-" * 50)
total_train = len(train_entries) + len(eq_train)
print(f"{'random_disp':<30} {n_random:>8} {n_random/total_train*100:>9.1f}%")
print(f"{'shear+perturbation':<30} {n_shear_perturb:>8} {n_shear_perturb/total_train*100:>9.1f}%")
print(f"{'shear clean':<30} {n_shear_clean:>8} {n_shear_clean/total_train*100:>9.1f}%")
print(f"{'uniaxial':<30} {n_uniaxial:>8} {n_uniaxial/total_train*100:>9.1f}%")
print(f"{'near_harmonic 90-atom':<30} {n_nh_90:>8} {n_nh_90/total_train*100:>9.1f}%")
print(f"{'near_harmonic 120-atom (conv)':<30} {n_nh_120:>8} {n_nh_120/total_train*100:>9.1f}%")
print(f"{'equilibrium':<30} {len(eq_train):>8} {len(eq_train)/total_train*100:>9.1f}%")
print("-" * 50)
print(f"{'TOTAL':<30} {total_train:>8} {'100.0%':>10}")
print(f"\n=== FINAL FILE SUMMARY ===")
print(f"  train.xyz : {len(train_entries)} stratified + {len(eq_train)} eq = {total_train} total")
print(f"  test.xyz  : {len(test_entries)} stratified only")
print(f"\n  NOTE: No weight= keyword in any XYZ header")
print(f"        NEP treats all structures with equal weight=1.0 (default)")

Cell 3 data: 3068 structures OK

File Label                                     Total  Train   Test
---------------------------------------------------------------
near_harmonic/Bi2Te3                             120     96     24
near_harmonic/Bi2Te3_conv120                      15     12      3
near_harmonic/BiTeSb_20                           50     40     10
near_harmonic/BiTeSb_40                           50     40     10
near_harmonic/BiTeSb_60                           50     40     10
near_harmonic/BiTeSb_80                           50     40     10
near_harmonic/Sb2Te3                             120     96     24
near_harmonic/Sb2Te3_conv120                      15     12      3
random_disp/Bi2Te3                               200    160     40
random_disp/BiTeSb_20                            200    160     40
random_disp/BiTeSb_40                            200    160     40
random_disp/BiTeSb_60                            200    160     40
random_disp/BiTeSb_80           

In [ ]:
# # ============================================================
# # CELL 4 — stress convert + stratified split +
# #           equilibrium structures — selective F/V per data_type
# #
# # data_type logic:
# #   random_disp  → E + F only      (large 0.3A disp, stress unreliable)
# #   shear        → E + V only      (strain structures, stress is key)
# #   uniaxial     → E + V only      (strain structures, stress is key)
# #   equilibrium  → E + F + V all   (zero-force reference, stress anchor)
# #   near_eq      → E + F + V all   (harmonic regime, both meaningful)
# # ============================================================

# import re
# import random
# import os
# import numpy as np
# from collections import defaultdict
# from monty.serialization import loadfn

# # ------------------------------------------------------------------
# # Constants
# # ------------------------------------------------------------------
# FLIP_STRESS_SIGN  = True
# GPA_TO_EV_PER_A3  = 1.0 / 160.21766208
# N_EQ_COPIES       = 2     # repeat each eq structure N times

# # ------------------------------------------------------------------
# # 1. Stress conversion — GPa Voigt-6 → eV/Å³ 9-component
# # ------------------------------------------------------------------
# def convert_voigt6_gpa_to_ev_per_a3(voigt6):
#     factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
#     σxx, σyy, σzz, σyz, σxz, σxy = voigt6
#     return [
#         σxx*factor, σxy*factor, σxz*factor,
#         σxy*factor, σyy*factor, σyz*factor,
#         σxz*factor, σyz*factor, σzz*factor,
#     ]

# # ------------------------------------------------------------------
# # 2. Metadata parser
# # ------------------------------------------------------------------
# def get_strain_metadata(file_label):
#     tag   = file_label.split("/")[-1]
#     dtype = file_label.split("/")[0]
#     strain_pct   = None
#     perturbation = 0.0
#     direction    = None
#     if dtype == "random_disp":
#         perturbation = 0.3
#     elif dtype == "shear":
#         m_strain = re.search(r'(\d+)pct', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         m_ptb2 = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb2:
#             perturbation = float(m_ptb2.group(1))
#     elif dtype == "uniaxial":
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0
#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # 3. Sanity check on Cell 3 outputs
# # ------------------------------------------------------------------
# N = len(total_structures)
# assert len(total_energies) == N
# assert len(total_forces)   == N
# assert len(total_stresses) == N
# print(f"Cell 3 data: {N} structures OK")

# # ------------------------------------------------------------------
# # 4. Pack entries from Cell 3 data
# # ------------------------------------------------------------------
# entries = []
# for i, d in enumerate(total_data):
#     config_type                         = d["_eref_key"]
#     data_type                           = d["_file_label"].split("/")[0]
#     strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
#     entries.append({
#         "structure"   : total_structures[i],
#         "_file_label" : d["_file_label"],
#         "config_type" : config_type,
#         "data_type"   : data_type,
#         "strain_pct"  : strain_pct,
#         "perturbation": perturbation,
#         "direction"   : direction,
#         "outputs": {
#             "energy": total_energies[i],
#             "forces": total_forces[i],
#             "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])
#         }
#     })

# # ------------------------------------------------------------------
# # 5. Stratified 80/20 split — 20% from each individual file
# # ------------------------------------------------------------------
# random.seed(42)

# strata = defaultdict(list)
# for e in entries:
#     strata[e["_file_label"]].append(e)

# train_entries = []
# test_entries  = []

# print(f"\n{'File Label':<40} {'Total':>6} {'Train':>6} {'Test':>6}")
# print("-" * 58)

# for key in sorted(strata.keys()):
#     group = strata[key]
#     random.shuffle(group)
#     n_test  = max(1, int(round(len(group) * 0.20)))
#     n_train = len(group) - n_test
#     test_entries  += group[:n_test]
#     train_entries += group[n_test:]
#     print(f"{key:<40} {len(group):>6} {n_train:>6} {n_test:>6}")

# print("-" * 58)
# print(f"{'TOTAL':<40} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
# print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
#       f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# # ------------------------------------------------------------------
# # 6. Load equilibrium structures
# #    Each comp gets its OWN eref_90 subtracted
# # ------------------------------------------------------------------
# eq_base  = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/equib_structure/"

# eq_files = {
#     "Bi2Te3"  : eq_base + "Bi2Te3_equilibrium.json",
#     "BiSbTe20": eq_base + "BiSbTe20_equilibrium.json",
#     "BiSbTe40": eq_base + "BiSbTe40_equilibrium.json",
#     "BiSbTe60": eq_base + "BiSbTe60_equilibrium.json",
#     "BiSbTe80": eq_base + "BiSbTe80_equilibrium.json",
#     "Sb2Te3"  : eq_base + "Sb2Te3_equilibrium.json",
# }

# eq_entries = []
# print(f"\n--- Equilibrium reference subtraction ---")
# print(f"{'Comp':<12} {'E_raw(eV)':>14} {'eref_90(eV)':>14} {'E_coh(eV)':>12} {'eV/atom':>10}")
# print("-" * 66)

# for comp, path in eq_files.items():
#     data   = loadfn(path)
#     d      = data[0]
#     struct = d["structure"]
#     e_raw  = d["outputs"]["energy"]
#     e_ref  = eref_90[comp]
#     e_coh  = e_raw - e_ref
#     forces = d["outputs"]["forces"]
#     stress = convert_voigt6_gpa_to_ev_per_a3(d["outputs"]["virial_stress"])

#     eq_entries.append({
#         "structure"   : struct,
#         "_file_label" : f"equilibrium/{comp}",
#         "config_type" : comp,
#         "data_type"   : "equilibrium",   # E + F + V all three
#         "strain_pct"  : None,
#         "perturbation": 0.0,
#         "direction"   : None,
#         "outputs"     : {"energy": e_coh, "forces": forces, "stress": stress},
#     })
#     print(f"{comp:<12} {e_raw:>14.4f} {e_ref:>14.4f} {e_coh:>12.4f} {e_coh/len(struct):>10.4f}")

# print("-" * 66)
# eq_mean    = sum(e['outputs']['energy']/len(e['structure']) for e in eq_entries)/len(eq_entries)
# train_mean = total_energies.mean()/90
# print(f"Eq mean    : {eq_mean:.4f} eV/atom")
# print(f"Train mean : {train_mean:.4f} eV/atom")

# eq_train = eq_entries * N_EQ_COPIES
# print(f"\n{len(eq_entries)} eq structs × {N_EQ_COPIES} copies = {len(eq_train)} for train")

# # ------------------------------------------------------------------
# # 7. XYZ writer — selective force/stress per data_type
# # ------------------------------------------------------------------
# def get_include_flags(data_type):
#     """
#     Returns (include_forces, include_stress, props_str) for each data_type.

#     random_disp  → E + F only      (0.3A disp, stress unreliable)
#     shear        → E + V only      (stress is the training target)
#     uniaxial     → E + V only      (stress is the training target)
#     equilibrium  → E + F + V all   (zero-force reference + stress anchor)
#     near_eq      → E + F + V all   (harmonic regime, both meaningful)
#     """
#     if data_type == "random_disp":
#         return True, False, 'properties=species:S:1:pos:R:3:forces:R:3'
#     elif data_type in ("shear", "uniaxial"):
#         return False, True,  'properties=species:S:1:pos:R:3'
#     elif data_type in ("equilibrium", "near_eq"):
#         return True,  True,  'properties=species:S:1:pos:R:3:forces:R:3'
#     else:
#         # fallback — include everything
#         return True,  True,  'properties=species:S:1:pos:R:3:forces:R:3'


# def write_entry_xyz(f, struct, out, config_type, data_type,
#                     strain_pct, perturbation, direction):
#     n = len(struct)
#     f.write(f"{n}\n")

#     lat_str    = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
#     e_str      = f"{out['energy']:.12f}"
#     s_str      = " ".join(f"{x:.12f}" for x in out['stress'])
#     strain_str = f"strain_pct={strain_pct}" if strain_pct is not None else "strain_pct=None"
#     ptb_str    = f"perturbation={perturbation}A"
#     dir_str    = f"direction={direction}"    if direction  is not None else "direction=None"

#     include_forces, include_stress, props = get_include_flags(data_type)

#     # build header
#     header = [f'lattice="{lat_str}"', f'energy={e_str}']
#     if include_stress:
#         header.append(f'stress="{s_str}"')
#     header += [
#         f'config_type={config_type}',
#         f'data_type={data_type}',
#         strain_str, ptb_str, dir_str,
#         'energy_units=eV',
#         'forces_units=eV/Ang',
#         'stress_units=eV/Ang3',
#         props,
#     ]
#     f.write(" ".join(header) + "\n")

#     # write atom lines
#     coords = struct.cart_coords
#     for idx, site in enumerate(struct.sites):
#         sym      = site.species.elements[0].symbol
#         x, y, z  = coords[idx]
#         if include_forces:
#             fx, fy, fz = out['forces'][idx]
#             f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")
#         else:
#             f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}\n")

# # ------------------------------------------------------------------
# # 8. Dataset writer
# # ------------------------------------------------------------------
# def write_dataset(fname, data, include_eq=False):
#     counts = defaultdict(int)
#     with open(fname, 'w') as f:
#         for e in data:
#             write_entry_xyz(
#                 f, e['structure'], e['outputs'],
#                 e['config_type'], e['data_type'],
#                 e['strain_pct'],  e['perturbation'], e['direction'],
#             )
#             counts[e['data_type']] += 1
#         if include_eq:
#             for e in eq_train:
#                 write_entry_xyz(
#                     f, e['structure'], e['outputs'],
#                     e['config_type'], e['data_type'],
#                     e['strain_pct'],  e['perturbation'], e['direction'],
#                 )
#                 counts[e['data_type']] += 1

#     total = sum(counts.values())

#     # labels for what's included
#     label_map = {
#         'random_disp' : 'E + F only',
#         'shear'       : 'E + V only',
#         'uniaxial'    : 'E + V only',
#         'equilibrium' : 'E + F + V',
#         'near_eq'     : 'E + F + V',
#     }
#     print(f"\nWrote {total} entries → '{fname}'")
#     print(f"  {'data_type':<20} {'structs':>8}  {'trains on':>12}")
#     print(f"  {'-'*44}")
#     for dt, n in sorted(counts.items()):
#         lbl = label_map.get(dt, 'E + F + V')
#         print(f"  {dt:<20} {n:>8}  {lbl:>12}")

# # ------------------------------------------------------------------
# # 9. Write files
# # ------------------------------------------------------------------
# out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"
# os.makedirs(out_dir, exist_ok=True)

# write_dataset(f"{out_dir}/train.xyz", train_entries, include_eq=True)
# write_dataset(f"{out_dir}/test.xyz",  test_entries,  include_eq=False)

# # ------------------------------------------------------------------
# # 10. Final summary
# # ------------------------------------------------------------------
# n_random        = sum(1 for e in train_entries if e["data_type"]=="random_disp")
# n_shear_perturb = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]!=0.0)
# n_shear_clean   = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]==0.0)
# n_uniaxial      = sum(1 for e in train_entries if e["data_type"]=="uniaxial")
# total_train     = len(train_entries) + len(eq_train)

# print(f"\n{'='*62}")
# print(f"DATASET COMPOSITION")
# print(f"{'='*62}")
# print(f"{'Data type':<25} {'Structs':>8} {'Frac':>8}  {'Trains on':>12}")
# print(f"{'-'*62}")
# rows = [
#     ("random_disp",        n_random,        "E + F only"),
#     ("shear+perturbation", n_shear_perturb, "E + V only"),
#     ("shear clean",        n_shear_clean,   "E + V only"),
#     ("uniaxial",           n_uniaxial,      "E + V only"),
#     ("equilibrium",        len(eq_train),   "E + F + V "),
# ]
# for name, n, lbl in rows:
#     print(f"  {name:<23} {n:>8} {n/total_train*100:>7.1f}%  {lbl:>12}")
# print(f"{'-'*62}")
# print(f"  {'TOTAL':<23} {total_train:>8} {'100.0%':>8}")
# print(f"\n{'='*62}")
# print(f"FILE SUMMARY")
# print(f"{'='*62}")
# print(f"  train.xyz : {len(train_entries)} stratified + {len(eq_train)} eq = {total_train} total")
# print(f"  test.xyz  : {len(test_entries)} stratified only")
# print(f"\n  Selective training logic:")
# print(f"    random_disp  → E + F only  (stress excluded — unreliable at 0.3A)")
# print(f"    shear        → E + V only  (forces excluded — stress is target)")
# print(f"    uniaxial     → E + V only  (forces excluded — stress is target)")
# print(f"    equilibrium  → E + F + V   (zero-force ref + stress anchor)")
# print(f"    near_eq      → E + F + V   (harmonic regime, both meaningful)")

Cell 3 data: 2598 structures OK

File Label                                Total  Train   Test
----------------------------------------------------------
random_disp/Bi2Te3                          200    160     40
random_disp/BiTeSb_20                       200    160     40
random_disp/BiTeSb_40                       200    160     40
random_disp/BiTeSb_60                       200    160     40
random_disp/BiTeSb_80                       200    160     40
random_disp/Sb2Te3                          200    160     40
shear/20_1pct_0.3ptb                         50     40     10
shear/20_1pct_0ptb                           50     40     10
shear/20_3pct_0.3ptb                         50     40     10
shear/20_3pct_0ptb                           50     40     10
shear/40_1pct_0.3ptb                         50     40     10
shear/40_1pct_0ptb                           50     40     10
shear/40_3pct_0.3ptb                         50     40     10
shear/40_3pct_0ptb                      

In [ ]:
# # ============================================================
# # CELL 4 — Convert stress, stratified 80/20 split per file,
# #           write NEP train/test XYZ
# # ============================================================

# import re
# import random
# from collections import defaultdict

# # ------------------------------------------------------------------
# # Constants
# # ------------------------------------------------------------------
# FLIP_STRESS_SIGN = True
# GPA_TO_EV_PER_A3 = 1.0 / 160.21766208

# # ------------------------------------------------------------------
# # Stress conversion
# # ------------------------------------------------------------------
# def convert_voigt6_gpa_to_ev_per_a3(voigt6):
#     factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
#     σxx, σyy, σzz, σyz, σxz, σxy = voigt6
#     return [
#         σxx*factor, σxy*factor, σxz*factor,
#         σxy*factor, σyy*factor, σyz*factor,
#         σxz*factor, σyz*factor, σzz*factor,
#     ]

# # ------------------------------------------------------------------
# # Metadata parser
# # ------------------------------------------------------------------
# def get_strain_metadata(file_label):
#     tag   = file_label.split("/")[-1]
#     dtype = file_label.split("/")[0]
#     strain_pct   = None
#     perturbation = 0.0
#     direction    = None
#     if dtype == "random_disp":
#         perturbation = 0.3
#     elif dtype == "shear":
#         m_strain = re.search(r'(\d+)pct', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb:
#             perturbation = float(m_ptb.group(1))
#     elif dtype == "uniaxial":
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0
#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # Sanity check
# # ------------------------------------------------------------------
# N = len(total_structures)
# assert len(total_energies) == N
# assert len(total_forces)   == N
# assert len(total_stresses) == N

# # ------------------------------------------------------------------
# # Pack entries — carry _file_label for stratification
# # ------------------------------------------------------------------
# entries = []
# for i, d in enumerate(total_data):
#     config_type                         = d["_eref_key"]
#     data_type                           = d["_file_label"].split("/")[0]
#     strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
#     entries.append({
#         "structure"   : total_structures[i],
#         "_file_label" : d["_file_label"],      # key for stratification
#         "config_type" : config_type,
#         "data_type"   : data_type,
#         "strain_pct"  : strain_pct,
#         "perturbation": perturbation,
#         "direction"   : direction,
#         "outputs": {
#             "energy": total_energies[i],
#             "forces": total_forces[i],
#             "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])
#         }
#     })

# # ------------------------------------------------------------------
# # Stratified 80/20 split — 20% from each individual file
# # ------------------------------------------------------------------
# random.seed(42)

# strata = defaultdict(list)
# for e in entries:
#     strata[e["_file_label"]].append(e)

# train_entries = []
# test_entries  = []

# print(f"\n{'File Label':<40} {'Total':>6} {'Train':>6} {'Test':>6}")
# print("-" * 58)

# for key in sorted(strata.keys()):
#     group = strata[key]
#     random.shuffle(group)
#     n_test  = max(1, int(round(len(group) * 0.20)))
#     n_train = len(group) - n_test
#     test_entries  += group[:n_test]
#     train_entries += group[n_test:]
#     print(f"{key:<40} {len(group):>6} {n_train:>6} {n_test:>6}")

# print("-" * 58)
# print(f"{'TOTAL':<40} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
# print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
#       f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# # ------------------------------------------------------------------
# # XYZ writer
# # ------------------------------------------------------------------
# def write_entry_xyz(f, struct, out, config_type, data_type,
#                     strain_pct, perturbation, direction):
#     n = len(struct)
#     f.write(f"{n}\n")
#     lat_str    = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
#     e_str      = f"{out['energy']:.12f}"
#     s_str      = " ".join(f"{x:.12f}" for x in out['stress'])
#     strain_str = f"strain_pct={strain_pct}"      if strain_pct is not None else "strain_pct=None"
#     ptb_str    = f"perturbation={perturbation}A"
#     dir_str    = f"direction={direction}"         if direction  is not None else "direction=None"
#     header = [
#         f'lattice="{lat_str}"',
#         f'energy={e_str}',
#         f'stress="{s_str}"',
#         f'config_type={config_type}',
#         f'data_type={data_type}',
#         strain_str, ptb_str, dir_str,
#         'energy_units=eV',
#         'forces_units=eV/Ang',
#         'stress_units=eV/Ang3',
#         'properties=species:S:1:pos:R:3:forces:R:3'
#     ]
#     f.write(" ".join(header) + "\n")
#     coords = struct.cart_coords
#     for idx, (fx, fy, fz) in enumerate(out['forces']):
#         sym      = struct.sites[idx].species.elements[0].symbol
#         x, y, z  = coords[idx]
#         f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")

# def write_dataset(fname, data):
#     with open(fname, 'w') as f:
#         for e in data:
#             write_entry_xyz(
#                 f, e['structure'], e['outputs'],
#                 e['config_type'], e['data_type'],
#                 e['strain_pct'],  e['perturbation'], e['direction']
#             )
#     print(f"Wrote {len(data)} entries → '{fname}'")

# # ------------------------------------------------------------------
# # Write files
# # ------------------------------------------------------------------
# out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"
# write_dataset(f"{out_dir}/train.xyz", train_entries)
# write_dataset(f"{out_dir}/test.xyz",  test_entries)


File Label                                Total  Train   Test
----------------------------------------------------------
random_disp/Bi2Te3                          200    160     40
random_disp/BiTeSb_20                       200    160     40
random_disp/BiTeSb_40                       200    160     40
random_disp/BiTeSb_60                       200    160     40
random_disp/BiTeSb_80                       200    160     40
random_disp/Sb2Te3                          200    160     40
shear/20_1pct_0.3ptb                         50     40     10
shear/20_1pct_0ptb                           50     40     10
shear/20_3pct_0.3ptb                         50     40     10
shear/20_3pct_0ptb                           50     40     10
shear/40_1pct_0.3ptb                         50     40     10
shear/40_1pct_0ptb                           50     40     10
shear/40_3pct_0.3ptb                         50     40     10
shear/40_3pct_0ptb                           50     40     10
shear/60_1

In [ ]:
# # ============================================================
# # CELL 4 — Full: stress convert + stratified split +
# #           equilibrium structures + all weights
# # ============================================================

# # ============================================================
# # WEIGHT SCHEME — Detailed Justification
# # ============================================================
# #
# # BACKGROUND
# # ----------
# # GPUMD NEP loss function:
# #   L = λ_e * RMSE(energy) + λ_f * RMSE(forces) + λ_v * RMSE(virial)
# #
# # The per-structure `weight` keyword multiplies the contribution
# # of that structure to ALL three loss terms (energy, force, virial).
# # Default weight = 1.0 for all structures.
# # Only RATIOS matter — weight=2.0 means 2× more important than weight=1.0.
# #
# # DATASET COMPOSITION (train set)
# # --------------------------------
# #   random_disp   : 960  structs  (6 comp × 160)  — bulk of training data
# #   shear+perturb : 720  structs  (24 files × 40)  — shear + atomic noise
# #   shear clean   : 320  structs  (8 files × 40)   — pure lattice shear
# #   uniaxial      : 162  structs  (18 files × 9)   — strain along X/Y/Z
# #   equilibrium   : 30   structs  (6 comp × 5)     — fully relaxed, zero-force
# #   TOTAL         : 2112 structs
# #
# # WITHOUT WEIGHTING — what happens?
# # ----------------------------------
# # NEP minimises average RMSE across all structures equally.
# # Since random_disp dominates (960/2112 = 45%), NEP will
# # preferentially fit large-force, high-energy displaced configs.
# # Uniaxial (162 structs, 7.7%) and equilibrium (30 structs, 1.4%)
# # get statistically underrepresented → poor Ecoh and Cij.
# #
# # WEIGHT ASSIGNMENT RATIONALE
# # ----------------------------
# #
# # weight = 1.0 → random_disp
# #   - Largest group (960 structs) — already statistically dominant
# #   - Contains large forces (mean ~3.7 eV/Å) → drives force RMSE
# #   - No boosting needed — NEP naturally fits these well
# #   - Baseline reference weight
# #
# # weight = 1.0 → shear + perturbation (0.3Å atomic noise)
# #   - 720 structs — well represented
# #   - Mixed signal: lattice shear + atomic displacement noise
# #   - Atomic noise makes virial signal noisier → don't overweight
# #   - Same weight as random_disp is appropriate
# #
# # weight = 2.0 → shear clean (0 perturbation)
# #   - 320 structs — pure lattice shear, no atomic noise
# #   - Cleaner virial/stress signal → better for learning Cij
# #   - Slightly boosted to emphasise clean stress data
# #   - 2× means each clean shear struct ≈ 2 random_disp structs
# #
# # weight = 5.0 → uniaxial strain
# #   - Only 162 structs (9 per file × 18 files) — smallest group
# #   - MOST CRITICAL for Cij: uniaxial strain directly probes
# #     C11, C33 (axial) and C12, C13 (transverse response)
# #   - Without boosting: 162/2112 = 7.7% → statistically weak
# #   - With weight=5: effective count = 162×5 = 810 → 28% effective
# #   - This compensates for the count imbalance vs random_disp
# #   - Rule of thumb: weight ≈ N_dominant / N_minority
# #     = 960 / 162 ≈ 6 → we use 5 (conservative)
# #
# # weight = 10.0 → equilibrium structures
# #   - Only 6 unique structures (×5 copies = 30 total)
# #   - MOST CRITICAL for Ecoh: zero-force, zero-stress reference point
# #   - Without boosting: 30/2112 = 1.4% → essentially invisible to NEP
# #   - With weight=10: effective count = 30×10 = 300 → ~12% effective
# #   - Ensures NEP anchors the energy minimum correctly
# #   - Forces are ~zero (MaxF < 0.1 eV/Å) → contributes mainly to
# #     energy term, not force term
# #   - High weight justified because: (a) very few structures,
# #     (b) physically most important reference point,
# #     (c) NEP must reproduce Ecoh accurately for thermodynamics
# #
# # EFFECTIVE COUNT AFTER WEIGHTING
# # ---------------------------------
# #   Data type          Structs   Weight   Effective
# #   random_disp          960      1.0       960
# #   shear+perturb        720      1.0       720
# #   shear clean          320      2.0       640
# #   uniaxial             162      5.0       810
# #   equilibrium           30     10.0       300
# #   TOTAL               2112               3430
# #
# #   Effective fractions:
# #   random_disp  : 960/3430  = 28.0%  (was 45.5% unweighted)
# #   shear+perturb: 720/3430  = 21.0%  (was 34.1% unweighted)
# #   shear clean  : 640/3430  = 18.7%  (was 15.1% unweighted)
# #   uniaxial     : 810/3430  = 23.6%  (was  7.7% unweighted)  ← boosted
# #   equilibrium  : 300/3430  =  8.7%  (was  1.4% unweighted)  ← boosted
# #
# # HOW TO TUNE
# # ------------
# # 1. Start training → check loss curve convergence
# # 2. After training → compute Ecoh and Cij vs DFT
# # 3. If Ecoh error > 5 meV/atom  → increase WEIGHT_EQUILIBRIUM (try 15-20)
# # 4. If Cij error  > 5 GPa       → increase WEIGHT_UNIAXIAL (try 8-10)
# # 5. If force RMSE gets worse    → reduce eq/uniaxial weights
# # 6. If energy RMSE gets worse   → reduce eq weight or N_EQ_COPIES
# # 7. Run 3 independent seeds → pick best model by test RMSE
# #
# # REFERENCE
# # ----------
# # GPUMD docs: weight is optional keyword in train.xyz header.
# # "weight=relative_weight gives the relative weight for the
# #  current structure in the total loss function." — gpumd.org
# # No published benchmark exists for optimal values — empirical.
# # ============================================================

# import re
# import random
# import os
# from collections import defaultdict
# from monty.serialization import loadfn

# # ------------------------------------------------------------------
# # Weight scheme — tunable here in one place
# # ------------------------------------------------------------------
# FLIP_STRESS_SIGN     = True
# GPA_TO_EV_PER_A3     = 1.0 / 160.21766208

# WEIGHT_RANDOM_DISP   = 1.0   # 960 structures  — normal weight
# WEIGHT_SHEAR_PERTURB = 1.0   # 720 structures  — normal weight
# WEIGHT_SHEAR_CLEAN   = 2.0   # 320 structures  — clean virial signal
# WEIGHT_UNIAXIAL      = 5.0   # 162 structures  — Cij critical
# WEIGHT_EQUILIBRIUM   = 10.0  # 6×5=30 structs  — anchor Ecoh
# N_EQ_COPIES          = 5     # repeat each eq structure N times

# # ------------------------------------------------------------------
# # 1. Stress conversion — GPa Voigt-6 → eV/Å³ 9-component
# # ------------------------------------------------------------------
# def convert_voigt6_gpa_to_ev_per_a3(voigt6):
#     factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
#     σxx, σyy, σzz, σyz, σxz, σxy = voigt6
#     return [
#         σxx*factor, σxy*factor, σxz*factor,
#         σxy*factor, σyy*factor, σyz*factor,
#         σxz*factor, σyz*factor, σzz*factor,
#     ]

# # ------------------------------------------------------------------
# # 2. Metadata parser
# # ------------------------------------------------------------------
# def get_strain_metadata(file_label):
#     tag   = file_label.split("/")[-1]
#     dtype = file_label.split("/")[0]
#     strain_pct   = None
#     perturbation = 0.0
#     direction    = None
#     if dtype == "random_disp":
#         perturbation = 0.3
#     elif dtype == "shear":
#         m_strain = re.search(r'(\d+)pct', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb:
#             perturbation = float(m_ptb.group(1))
#     elif dtype == "uniaxial":
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0
#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # 3. Weight assignment by data type
# # ------------------------------------------------------------------
# def get_weight(data_type, perturbation=0.0):
#     if data_type == "equilibrium":
#         return WEIGHT_EQUILIBRIUM
#     elif data_type == "uniaxial":
#         return WEIGHT_UNIAXIAL
#     elif data_type == "shear":
#         if perturbation == 0.0:
#             return WEIGHT_SHEAR_CLEAN
#         else:
#             return WEIGHT_SHEAR_PERTURB
#     else:
#         return WEIGHT_RANDOM_DISP

# # ------------------------------------------------------------------
# # 4. Sanity check on Cell 3 outputs
# # ------------------------------------------------------------------
# N = len(total_structures)
# assert len(total_energies) == N
# assert len(total_forces)   == N
# assert len(total_stresses) == N
# print(f"Cell 3 data: {N} structures OK")

# # ------------------------------------------------------------------
# # 5. Pack entries from Cell 3 data
# # ------------------------------------------------------------------
# entries = []
# for i, d in enumerate(total_data):
#     config_type                         = d["_eref_key"]
#     data_type                           = d["_file_label"].split("/")[0]
#     strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
#     entries.append({
#         "structure"   : total_structures[i],
#         "_file_label" : d["_file_label"],
#         "config_type" : config_type,
#         "data_type"   : data_type,
#         "strain_pct"  : strain_pct,
#         "perturbation": perturbation,
#         "direction"   : direction,
#         "outputs": {
#             "energy": total_energies[i],
#             "forces": total_forces[i],
#             "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])
#         }
#     })

# # ------------------------------------------------------------------
# # 6. Stratified 80/20 split — 20% from each individual file
# # ------------------------------------------------------------------
# random.seed(42)

# strata = defaultdict(list)
# for e in entries:
#     strata[e["_file_label"]].append(e)

# train_entries = []
# test_entries  = []

# print(f"\n{'File Label':<40} {'Total':>6} {'Train':>6} {'Test':>6} {'Weight':>8}")
# print("-" * 66)

# for key in sorted(strata.keys()):
#     group = strata[key]
#     random.shuffle(group)
#     n_test  = max(1, int(round(len(group) * 0.20)))
#     n_train = len(group) - n_test
#     test_entries  += group[:n_test]
#     train_entries += group[n_test:]
#     sample = group[0]
#     w      = get_weight(sample["data_type"], sample.get("perturbation", 0.0))
#     print(f"{key:<40} {len(group):>6} {n_train:>6} {n_test:>6} {w:>8.1f}")

# print("-" * 66)
# print(f"{'TOTAL':<40} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
# print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
#       f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# # ------------------------------------------------------------------
# # 7. Load equilibrium structures
# #    loadfn already deserializes pymatgen Structure — use directly
# #    Each comp gets its OWN eref_90 subtracted
# # ------------------------------------------------------------------
# eq_base  = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/equib_structure/"

# eq_files = {
#     "Bi2Te3"  : eq_base + "Bi2Te3_equilibrium.json",
#     "BiSbTe20": eq_base + "BiSbTe20_equilibrium.json",
#     "BiSbTe40": eq_base + "BiSbTe40_equilibrium.json",
#     "BiSbTe60": eq_base + "BiSbTe60_equilibrium.json",
#     "BiSbTe80": eq_base + "BiSbTe80_equilibrium.json",
#     "Sb2Te3"  : eq_base + "Sb2Te3_equilibrium.json",
# }

# eq_entries = []
# print(f"\n--- Equilibrium reference subtraction ---")
# print(f"{'Comp':<12} {'E_raw(eV)':>14} {'eref_90(eV)':>14} {'E_coh(eV)':>12} {'eV/atom':>10}")
# print("-" * 66)

# for comp, path in eq_files.items():
#     data   = loadfn(path)
#     d      = data[0]
#     struct = d["structure"]      # loadfn already gives Structure object directly
#     e_raw  = d["outputs"]["energy"]
#     e_ref  = eref_90[comp]       # each comp gets its OWN isolated-atom reference
#     e_coh  = e_raw - e_ref       # cohesive energy — consistent with training data
#     forces = d["outputs"]["forces"]
#     stress = convert_voigt6_gpa_to_ev_per_a3(d["outputs"]["virial_stress"])

#     eq_entries.append({
#         "structure"   : struct,
#         "_file_label" : f"equilibrium/{comp}",
#         "config_type" : comp,
#         "data_type"   : "equilibrium",
#         "strain_pct"  : None,
#         "perturbation": 0.0,
#         "direction"   : None,
#         "outputs"     : {"energy": e_coh, "forces": forces, "stress": stress},
#     })
#     print(f"{comp:<12} {e_raw:>14.4f} {e_ref:>14.4f} {e_coh:>12.4f} {e_coh/len(struct):>10.4f}")

# print("-" * 66)
# eq_mean    = sum(e['outputs']['energy']/len(e['structure']) for e in eq_entries)/len(eq_entries)
# train_mean = total_energies.mean()/90
# print(f"Eq mean    : {eq_mean:.4f} eV/atom")
# print(f"Train mean : {train_mean:.4f} eV/atom  ← should be close")

# eq_train = eq_entries * N_EQ_COPIES
# print(f"\n{len(eq_entries)} eq structs × {N_EQ_COPIES} copies = {len(eq_train)} for train")

# # ------------------------------------------------------------------
# # 8. XYZ writer with weight
# # ------------------------------------------------------------------
# def write_entry_xyz(f, struct, out, config_type, data_type,
#                     strain_pct, perturbation, direction, weight=1.0):
#     n = len(struct)
#     f.write(f"{n}\n")
#     lat_str    = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
#     e_str      = f"{out['energy']:.12f}"
#     s_str      = " ".join(f"{x:.12f}" for x in out['stress'])
#     strain_str = f"strain_pct={strain_pct}"      if strain_pct is not None else "strain_pct=None"
#     ptb_str    = f"perturbation={perturbation}A"
#     dir_str    = f"direction={direction}"         if direction  is not None else "direction=None"
#     header = [
#         f'lattice="{lat_str}"',
#         f'energy={e_str}',
#         f'stress="{s_str}"',
#         f'weight={weight}',
#         f'config_type={config_type}',
#         f'data_type={data_type}',
#         strain_str, ptb_str, dir_str,
#         'energy_units=eV',
#         'forces_units=eV/Ang',
#         'stress_units=eV/Ang3',
#         'properties=species:S:1:pos:R:3:forces:R:3'
#     ]
#     f.write(" ".join(header) + "\n")
#     coords = struct.cart_coords
#     for idx, (fx, fy, fz) in enumerate(out['forces']):
#         sym      = struct.sites[idx].species.elements[0].symbol
#         x, y, z  = coords[idx]
#         f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")

# # ------------------------------------------------------------------
# # 9. Dataset writer
# # ------------------------------------------------------------------
# def write_dataset(fname, data, include_eq=False):
#     with open(fname, 'w') as f:
#         for e in data:
#             w = get_weight(e["data_type"], e.get("perturbation", 0.0))
#             write_entry_xyz(
#                 f, e['structure'], e['outputs'],
#                 e['config_type'], e['data_type'],
#                 e['strain_pct'],  e['perturbation'], e['direction'],
#                 weight=w
#             )
#         if include_eq:
#             for e in eq_train:
#                 write_entry_xyz(
#                     f, e['structure'], e['outputs'],
#                     e['config_type'], e['data_type'],
#                     e['strain_pct'],  e['perturbation'], e['direction'],
#                     weight=WEIGHT_EQUILIBRIUM
#                 )
#     total = len(data) + (len(eq_train) if include_eq else 0)
#     print(f"Wrote {total} entries → '{fname}'")

# # ------------------------------------------------------------------
# # 10. Write files
# # ------------------------------------------------------------------
# out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"
# os.makedirs(out_dir, exist_ok=True)

# write_dataset(f"{out_dir}/train.xyz", train_entries, include_eq=True)
# write_dataset(f"{out_dir}/test.xyz",  test_entries,  include_eq=False)

# # ------------------------------------------------------------------
# # 11. Final summary
# # ------------------------------------------------------------------
# # count shear clean vs perturbed in train
# n_shear_clean  = sum(1 for e in train_entries
#                      if e["data_type"]=="shear" and e["perturbation"]==0.0)
# n_shear_perturb= sum(1 for e in train_entries
#                      if e["data_type"]=="shear" and e["perturbation"]!=0.0)
# n_random       = sum(1 for e in train_entries if e["data_type"]=="random_disp")
# n_uniaxial     = sum(1 for e in train_entries if e["data_type"]=="uniaxial")

# print(f"\n=== WEIGHT SCHEME + EFFECTIVE COUNTS ===")
# print(f"{'Data type':<25} {'Structs':>8} {'Weight':>8} {'Effective':>10}")
# print("-" * 55)
# print(f"{'random_disp':<25} {n_random:>8} {WEIGHT_RANDOM_DISP:>8.1f} {n_random*WEIGHT_RANDOM_DISP:>10.0f}")
# print(f"{'shear+perturbation':<25} {n_shear_perturb:>8} {WEIGHT_SHEAR_PERTURB:>8.1f} {n_shear_perturb*WEIGHT_SHEAR_PERTURB:>10.0f}")
# print(f"{'shear clean':<25} {n_shear_clean:>8} {WEIGHT_SHEAR_CLEAN:>8.1f} {n_shear_clean*WEIGHT_SHEAR_CLEAN:>10.0f}")
# print(f"{'uniaxial':<25} {n_uniaxial:>8} {WEIGHT_UNIAXIAL:>8.1f} {n_uniaxial*WEIGHT_UNIAXIAL:>10.0f}")
# print(f"{'equilibrium':<25} {len(eq_train):>8} {WEIGHT_EQUILIBRIUM:>8.1f} {len(eq_train)*WEIGHT_EQUILIBRIUM:>10.0f}")
# print("-" * 55)
# total_eff = (n_random*WEIGHT_RANDOM_DISP + n_shear_perturb*WEIGHT_SHEAR_PERTURB +
#              n_shear_clean*WEIGHT_SHEAR_CLEAN + n_uniaxial*WEIGHT_UNIAXIAL +
#              len(eq_train)*WEIGHT_EQUILIBRIUM)
# print(f"{'TOTAL EFFECTIVE':<25} {'':>8} {'':>8} {total_eff:>10.0f}")

# print(f"\n=== FINAL FILE SUMMARY ===")
# print(f"  train.xyz : {len(train_entries)} stratified + {len(eq_train)} eq = "
#       f"{len(train_entries)+len(eq_train)} total structures")
# print(f"  test.xyz  : {len(test_entries)} stratified only (no eq, unweighted eval)")

Cell 3 data: 2598 structures OK

File Label                                Total  Train   Test   Weight
------------------------------------------------------------------
random_disp/Bi2Te3                          200    160     40      1.0
random_disp/BiTeSb_20                       200    160     40      1.0
random_disp/BiTeSb_40                       200    160     40      1.0
random_disp/BiTeSb_60                       200    160     40      1.0
random_disp/BiTeSb_80                       200    160     40      1.0
random_disp/Sb2Te3                          200    160     40      1.0
shear/20_1pct_0.3ptb                         50     40     10      1.0
shear/20_1pct_0ptb                           50     40     10      2.0
shear/20_3pct_0.3ptb                         50     40     10      1.0
shear/20_3pct_0ptb                           50     40     10      2.0
shear/40_1pct_0.3ptb                         50     40     10      1.0
shear/40_1pct_0ptb                           50 

In [ ]:
# # ============================================================
# # CELL 4 — stress convert + stratified split +
# #           equilibrium structures — NO weights
# # ============================================================

# import re
# import random
# import os
# from collections import defaultdict
# from monty.serialization import loadfn

# # ------------------------------------------------------------------
# # Constants
# # ------------------------------------------------------------------
# FLIP_STRESS_SIGN  = True
# GPA_TO_EV_PER_A3  = 1.0 / 160.21766208
# N_EQ_COPIES       = 2     # repeat each eq structure N times

# # ------------------------------------------------------------------
# # 1. Stress conversion — GPa Voigt-6 → eV/Å³ 9-component
# # ------------------------------------------------------------------
# def convert_voigt6_gpa_to_ev_per_a3(voigt6):
#     factor = -GPA_TO_EV_PER_A3 if FLIP_STRESS_SIGN else GPA_TO_EV_PER_A3
#     σxx, σyy, σzz, σyz, σxz, σxy = voigt6
#     return [
#         σxx*factor, σxy*factor, σxz*factor,
#         σxy*factor, σyy*factor, σyz*factor,
#         σxz*factor, σyz*factor, σzz*factor,
#     ]

# # ------------------------------------------------------------------
# # 2. Metadata parser
# # ------------------------------------------------------------------
# def get_strain_metadata(file_label):
#     tag   = file_label.split("/")[-1]
#     dtype = file_label.split("/")[0]
#     strain_pct   = None
#     perturbation = 0.0
#     direction    = None
#     if dtype == "random_disp":
#         perturbation = 0.3
#     elif dtype == "shear":
#         m_strain = re.search(r'(\d+)pct', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb:
#             perturbation = float(m_ptb.group(1))
#     elif dtype == "uniaxial":
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0
#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # 3. Sanity check on Cell 3 outputs
# # ------------------------------------------------------------------
# N = len(total_structures)
# assert len(total_energies) == N
# assert len(total_forces)   == N
# assert len(total_stresses) == N
# print(f"Cell 3 data: {N} structures OK")

# # ------------------------------------------------------------------
# # 4. Pack entries from Cell 3 data
# # ------------------------------------------------------------------
# entries = []
# for i, d in enumerate(total_data):
#     config_type                         = d["_eref_key"]
#     data_type                           = d["_file_label"].split("/")[0]
#     strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
#     entries.append({
#         "structure"   : total_structures[i],
#         "_file_label" : d["_file_label"],
#         "config_type" : config_type,
#         "data_type"   : data_type,
#         "strain_pct"  : strain_pct,
#         "perturbation": perturbation,
#         "direction"   : direction,
#         "outputs": {
#             "energy": total_energies[i],
#             "forces": total_forces[i],
#             "stress": convert_voigt6_gpa_to_ev_per_a3(total_stresses[i])
#         }
#     })

# # ------------------------------------------------------------------
# # 5. Stratified 80/20 split — 20% from each individual file
# # ------------------------------------------------------------------
# random.seed(42)

# strata = defaultdict(list)
# for e in entries:
#     strata[e["_file_label"]].append(e)

# train_entries = []
# test_entries  = []

# print(f"\n{'File Label':<40} {'Total':>6} {'Train':>6} {'Test':>6}")
# print("-" * 58)

# for key in sorted(strata.keys()):
#     group = strata[key]
#     random.shuffle(group)
#     n_test  = max(1, int(round(len(group) * 0.20)))
#     n_train = len(group) - n_test
#     test_entries  += group[:n_test]
#     train_entries += group[n_test:]
#     print(f"{key:<40} {len(group):>6} {n_train:>6} {n_test:>6}")

# print("-" * 58)
# print(f"{'TOTAL':<40} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
# print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
#       f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# # ------------------------------------------------------------------
# # 6. Load equilibrium structures
# #    Each comp gets its OWN eref_90 subtracted
# # ------------------------------------------------------------------
# eq_base  = "/home/ashwani/BiSbTe_AlloyTransport/training_data_json_format/equib_structure/"

# eq_files = {
#     "Bi2Te3"  : eq_base + "Bi2Te3_equilibrium.json",
#     "BiSbTe20": eq_base + "BiSbTe20_equilibrium.json",
#     "BiSbTe40": eq_base + "BiSbTe40_equilibrium.json",
#     "BiSbTe60": eq_base + "BiSbTe60_equilibrium.json",
#     "BiSbTe80": eq_base + "BiSbTe80_equilibrium.json",
#     "Sb2Te3"  : eq_base + "Sb2Te3_equilibrium.json",
# }

# eq_entries = []
# print(f"\n--- Equilibrium reference subtraction ---")
# print(f"{'Comp':<12} {'E_raw(eV)':>14} {'eref_90(eV)':>14} {'E_coh(eV)':>12} {'eV/atom':>10}")
# print("-" * 66)

# for comp, path in eq_files.items():
#     data   = loadfn(path)
#     d      = data[0]
#     struct = d["structure"]
#     e_raw  = d["outputs"]["energy"]
#     e_ref  = eref_90[comp]
#     e_coh  = e_raw - e_ref
#     forces = d["outputs"]["forces"]
#     stress = convert_voigt6_gpa_to_ev_per_a3(d["outputs"]["virial_stress"])

#     eq_entries.append({
#         "structure"   : struct,
#         "_file_label" : f"equilibrium/{comp}",
#         "config_type" : comp,
#         "data_type"   : "equilibrium",
#         "strain_pct"  : None,
#         "perturbation": 0.0,
#         "direction"   : None,
#         "outputs"     : {"energy": e_coh, "forces": forces, "stress": stress},
#     })
#     print(f"{comp:<12} {e_raw:>14.4f} {e_ref:>14.4f} {e_coh:>12.4f} {e_coh/len(struct):>10.4f}")

# print("-" * 66)
# eq_mean    = sum(e['outputs']['energy']/len(e['structure']) for e in eq_entries)/len(eq_entries)
# train_mean = total_energies.mean()/90
# print(f"Eq mean    : {eq_mean:.4f} eV/atom")
# print(f"Train mean : {train_mean:.4f} eV/atom")

# eq_train = eq_entries * N_EQ_COPIES
# print(f"\n{len(eq_entries)} eq structs × {N_EQ_COPIES} copies = {len(eq_train)} for train")

# # ------------------------------------------------------------------
# # 7. XYZ writer — NO weight keyword in header
# # ------------------------------------------------------------------
# def write_entry_xyz(f, struct, out, config_type, data_type,
#                     strain_pct, perturbation, direction):
#     n = len(struct)
#     f.write(f"{n}\n")
#     lat_str    = " ".join(f"{v:.9f}" for v in struct.lattice.matrix.flatten())
#     e_str      = f"{out['energy']:.12f}"
#     s_str      = " ".join(f"{x:.12f}" for x in out['stress'])
#     strain_str = f"strain_pct={strain_pct}"  if strain_pct is not None else "strain_pct=None"
#     ptb_str    = f"perturbation={perturbation}A"
#     dir_str    = f"direction={direction}"     if direction  is not None else "direction=None"
#     header = [
#         f'lattice="{lat_str}"',
#         f'energy={e_str}',
#         f'stress="{s_str}"',
#         f'config_type={config_type}',   # ← no weight= keyword at all
#         f'data_type={data_type}',
#         strain_str, ptb_str, dir_str,
#         'energy_units=eV',
#         'forces_units=eV/Ang',
#         'stress_units=eV/Ang3',
#         'properties=species:S:1:pos:R:3:forces:R:3'
#     ]
#     f.write(" ".join(header) + "\n")
#     coords = struct.cart_coords
#     for idx, (fx, fy, fz) in enumerate(out['forces']):
#         sym      = struct.sites[idx].species.elements[0].symbol
#         x, y, z  = coords[idx]
#         f.write(f"{sym:<2s} {x: .6f} {y: .6f} {z: .6f}   {fx: .6f} {fy: .6f} {fz: .6f}\n")

# # ------------------------------------------------------------------
# # 8. Dataset writer
# # ------------------------------------------------------------------
# def write_dataset(fname, data, include_eq=False):
#     with open(fname, 'w') as f:
#         for e in data:
#             write_entry_xyz(
#                 f, e['structure'], e['outputs'],
#                 e['config_type'], e['data_type'],
#                 e['strain_pct'],  e['perturbation'], e['direction'],
#             )
#         if include_eq:
#             for e in eq_train:
#                 write_entry_xyz(
#                     f, e['structure'], e['outputs'],
#                     e['config_type'], e['data_type'],
#                     e['strain_pct'],  e['perturbation'], e['direction'],
#                 )
#     total = len(data) + (len(eq_train) if include_eq else 0)
#     print(f"Wrote {total} entries → '{fname}'")

# # ------------------------------------------------------------------
# # 9. Write files
# # ------------------------------------------------------------------
# out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_nep_format"
# os.makedirs(out_dir, exist_ok=True)

# write_dataset(f"{out_dir}/train.xyz", train_entries, include_eq=True)
# write_dataset(f"{out_dir}/test.xyz",  test_entries,  include_eq=False)

# # ------------------------------------------------------------------
# # 10. Final summary
# # ------------------------------------------------------------------
# n_random        = sum(1 for e in train_entries if e["data_type"]=="random_disp")
# n_shear_perturb = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]!=0.0)
# n_shear_clean   = sum(1 for e in train_entries if e["data_type"]=="shear" and e["perturbation"]==0.0)
# n_uniaxial      = sum(1 for e in train_entries if e["data_type"]=="uniaxial")

# print(f"\n=== DATASET COMPOSITION (no weights — all equal) ===")
# print(f"{'Data type':<25} {'Structs':>8} {'Fraction':>10}")
# print("-" * 45)
# total_train = len(train_entries) + len(eq_train)
# print(f"{'random_disp':<25} {n_random:>8} {n_random/total_train*100:>9.1f}%")
# print(f"{'shear+perturbation':<25} {n_shear_perturb:>8} {n_shear_perturb/total_train*100:>9.1f}%")
# print(f"{'shear clean':<25} {n_shear_clean:>8} {n_shear_clean/total_train*100:>9.1f}%")
# print(f"{'uniaxial':<25} {n_uniaxial:>8} {n_uniaxial/total_train*100:>9.1f}%")
# print(f"{'equilibrium':<25} {len(eq_train):>8} {len(eq_train)/total_train*100:>9.1f}%")
# print("-" * 45)
# print(f"{'TOTAL':<25} {total_train:>8} {'100.0%':>10}")
# print(f"\n=== FINAL FILE SUMMARY ===")
# print(f"  train.xyz : {len(train_entries)} stratified + {len(eq_train)} eq = {total_train} total")
# print(f"  test.xyz  : {len(test_entries)} stratified only")
# print(f"\n  NOTE: No weight= keyword in any XYZ header")
# print(f"        NEP treats all structures with equal weight=1.0 (default)")

Cell 3 data: 2598 structures OK

File Label                                Total  Train   Test
----------------------------------------------------------
random_disp/Bi2Te3                          200    160     40
random_disp/BiTeSb_20                       200    160     40
random_disp/BiTeSb_40                       200    160     40
random_disp/BiTeSb_60                       200    160     40
random_disp/BiTeSb_80                       200    160     40
random_disp/Sb2Te3                          200    160     40
shear/20_1pct_0.3ptb                         50     40     10
shear/20_1pct_0ptb                           50     40     10
shear/20_3pct_0.3ptb                         50     40     10
shear/20_3pct_0ptb                           50     40     10
shear/40_1pct_0.3ptb                         50     40     10
shear/40_1pct_0ptb                           50     40     10
shear/40_3pct_0.3ptb                         50     40     10
shear/40_3pct_0ptb                      

In [ ]:
# # ============================================================
# # CELL 5 — Stratified 80/20 split + Write MTP .cfg format
# #           for BiSbTe alloy dataset
# # ============================================================

# import re
# import random
# from collections import defaultdict
# from sklearn.model_selection import train_test_split

# # ------------------------------------------------------------------
# # Constants
# # ------------------------------------------------------------------
# FLIP_STRESS_SIGN = True
# GPA_TO_EV_PER_A3 = 0.0062415   # GPa → eV/Å³ (MTP convention uses volume-weighted)

# # ------------------------------------------------------------------
# # Species mapping — fixed order for NEP type consistency
# # Bi=0, Sb=1, Te=2  (3-component system)
# # ------------------------------------------------------------------
# SPECIES_TO_TYPE = {"Bi": 0, "Sb": 1, "Te": 2}
# print("Species → Type mapping:", SPECIES_TO_TYPE)

# # ------------------------------------------------------------------
# # Metadata parser
# # ------------------------------------------------------------------
# def get_strain_metadata(file_label):
#     tag   = file_label.split("/")[-1]
#     dtype = file_label.split("/")[0]
#     strain_pct   = None
#     perturbation = 0.0
#     direction    = None
#     if dtype == "random_disp":
#         perturbation = 0.3
#     elif dtype == "shear":
#         m_strain = re.search(r'(\d+)pct', tag)
#         if m_strain:
#             strain_pct = float(m_strain.group(1))
#         m_ptb = re.search(r'_(\d+\.?\d*)ptb', tag)
#         if m_ptb:
#             perturbation = float(m_ptb.group(1))
#     elif dtype == "uniaxial":
#         m_dir = re.search(r'_(X|Y|Z)$', tag)
#         if m_dir:
#             direction = m_dir.group(1)
#         perturbation = 0.0
#     return strain_pct, perturbation, direction

# # ------------------------------------------------------------------
# # Stress conversion: Voigt-6 GPa → eV (volume-weighted, MTP convention)
# # MTP PlusStress = stress (eV/Å³) × volume (Å³) = eV
# # Voigt order: [xx, yy, zz, yz, xz, xy]
# # Sign flip: atomate virial → MTP convention
# # ------------------------------------------------------------------
# def convert_stress_to_mtp(voigt6_gpa, volume):
#     sign = -1.0 if FLIP_STRESS_SIGN else 1.0
#     return [s * sign * GPA_TO_EV_PER_A3 * volume for s in voigt6_gpa]

# # ------------------------------------------------------------------
# # MTP .cfg writer — one structure block
# # ------------------------------------------------------------------
# def structure_to_cfg(structure, energy, forces, stresses_ev):
#     lattice   = structure.lattice.matrix
#     num_atoms = len(structure)
#     positions = structure.cart_coords

#     cfg  = "BEGIN_CFG\n"
#     cfg += f" Size\n    {num_atoms}\n"
#     cfg += " Supercell\n"
#     for row in lattice:
#         cfg += f"         {' '.join(f'{val:12.6f}' for val in row)}\n"

#     cfg += " AtomData:  id type       cartes_x      cartes_y      cartes_z           fx          fy          fz\n"
#     for i, (pos, force, site) in enumerate(zip(positions, forces, structure.sites), start=1):
#         sp        = str(site.species.elements[0])
#         atom_type = SPECIES_TO_TYPE[sp]
#         cfg += (
#             f"{i:13d}    {atom_type:<4d}"
#             f"{pos[0]:12.6f} {pos[1]:12.6f} {pos[2]:12.6f}   "
#             f"{force[0]:10.6f} {force[1]:10.6f} {force[2]:10.6f}\n"
#         )

#     cfg += " Energy\n"
#     cfg += f"        {energy:.12f}\n"

#     # PlusStress: xx yy zz yz xz xy  (in eV, volume-weighted)
#     cfg += " PlusStress:  xx          yy          zz          yz          xz          xy\n"
#     cfg += f"        {' '.join(f'{val:10.5f}' for val in stresses_ev)}\n"

#     cfg += " Feature   EFS_by\tVASP\n"
#     cfg += "END_CFG\n"
#     return cfg

# # ------------------------------------------------------------------
# # Sanity check
# # ------------------------------------------------------------------
# N = len(total_structures)
# assert len(total_energies) == N
# assert len(total_forces)   == N
# assert len(total_stresses) == N

# # ------------------------------------------------------------------
# # Pack entries — carry _file_label for stratification
# # ------------------------------------------------------------------
# entries = []
# for i, d in enumerate(total_data):
#     config_type                         = d["_eref_key"]
#     data_type                           = d["_file_label"].split("/")[0]
#     strain_pct, perturbation, direction = get_strain_metadata(d["_file_label"])
#     entries.append({
#         "structure"   : total_structures[i],
#         "_file_label" : d["_file_label"],
#         "config_type" : config_type,
#         "data_type"   : data_type,
#         "strain_pct"  : strain_pct,
#         "perturbation": perturbation,
#         "direction"   : direction,
#         "energy"      : total_energies_raw[i],   # raw DFT energy (NOT ref subtracted) for MTP
#         "forces"      : total_forces[i],
#         "stress"      : total_stresses[i],        # Voigt-6 GPa — converted per structure below
#     })

# # ------------------------------------------------------------------
# # NOTE on energy for MTP vs NEP:
# #   NEP  → needs reference-subtracted cohesive energy
# #   MTP  → needs raw DFT total energy (eV), no reference subtraction
# # So we use total_energies_raw here, not total_energies
# # ------------------------------------------------------------------

# # ------------------------------------------------------------------
# # Stratified 80/20 split — 20% from each individual file
# # ------------------------------------------------------------------
# random.seed(42)

# strata = defaultdict(list)
# for e in entries:
#     strata[e["_file_label"]].append(e)

# train_entries = []
# test_entries  = []

# print(f"\n{'File Label':<40} {'Total':>6} {'Train':>6} {'Test':>6}")
# print("-" * 58)

# for key in sorted(strata.keys()):
#     group = strata[key]
#     random.shuffle(group)
#     n_test  = max(1, int(round(len(group) * 0.20)))
#     n_train = len(group) - n_test
#     test_entries  += group[:n_test]
#     train_entries += group[n_test:]
#     print(f"{key:<40} {len(group):>6} {n_train:>6} {n_test:>6}")

# print("-" * 58)
# print(f"{'TOTAL':<40} {len(entries):>6} {len(train_entries):>6} {len(test_entries):>6}")
# print(f"\nSplit — Train: {len(train_entries)/len(entries)*100:.1f}%  "
#       f"Test:  {len(test_entries)/len(entries)*100:.1f}%")

# # ------------------------------------------------------------------
# # Write .cfg files
# # ------------------------------------------------------------------
# def write_cfg_dataset(fname, data):
#     with open(fname, 'w') as f:
#         for e in data:
#             struct      = e["structure"]
#             energy      = e["energy"]                                    # raw DFT eV
#             forces      = e["forces"]                                    # eV/Å
#             stress_ev   = convert_stress_to_mtp(e["stress"], struct.volume)  # eV
#             cfg_str     = structure_to_cfg(struct, energy, forces, stress_ev)
#             f.write(cfg_str)
#     print(f"Wrote {len(data)} structures → '{fname}'")

# out_dir = "/home/ashwani/BiSbTe_AlloyTransport/training_data_mtp_format"
# import os
# os.makedirs(out_dir, exist_ok=True)

# write_cfg_dataset(f"{out_dir}/train.cfg", train_entries)
# write_cfg_dataset(f"{out_dir}/test.cfg",  test_entries)

Species → Type mapping: {'Bi': 0, 'Sb': 1, 'Te': 2}

File Label                                Total  Train   Test
----------------------------------------------------------
random_disp/Bi2Te3                          200    160     40
random_disp/BiTeSb_20                       200    160     40
random_disp/BiTeSb_40                       200    160     40
random_disp/BiTeSb_60                       200    160     40
random_disp/BiTeSb_80                       200    160     40
random_disp/Sb2Te3                          200    160     40
shear/20_1pct_0.3ptb                         50     40     10
shear/20_1pct_0ptb                           50     40     10
shear/20_3pct_0.3ptb                         50     40     10
shear/20_3pct_0ptb                           50     40     10
shear/40_1pct_0.3ptb                         50     40     10
shear/40_1pct_0ptb                           50     40     10
shear/40_3pct_0.3ptb                         50     40     10
shear/40_3pct_0ptb  